# Preprocessing data

## Import data

One switch drives both ends. `RUNTIME = "auto"` detects Colab and mounts Drive;
`"local"` and `"colab"` force it either way. Everything downstream reads two
paths and nothing else:

| | `ROOT` (raw sessions) | `OUT_ROOT` (processed data + models) |
|---|---|---|
| local | `./data` | `./data` — gitignored, so artifacts never reach the repo |
| colab | `DRIVE_DATA` | `DRIVE_OUT` — survives the runtime dying |

The loader accepts either directory layout (`user7/` or Balabit's original
`user_7/`) and session files with or without a `.csv` extension, so the same
notebook reads your Drive copy and the local one.

In [ ]:
# ---------------------------------------------------------------------------
# Runtime: local checkout or Google Colab
# ---------------------------------------------------------------------------
# "auto"  -- Colab if google.colab is importable, otherwise local
# "local" -- force the repo checkout (data in ./data)
# "colab" -- force Drive, even where detection would say otherwise
RUNTIME = "auto"

# --- Colab only -------------------------------------------------------------
DRIVE_DATA = "/content/drive/MyDrive/trace-data/balabit"   # holds user*/session_*
DRIVE_OUT  = "/content/drive/MyDrive/trace-data"           # processed/ and models/ land here

# Drive is a FUSE mount: ~1700 small CSVs over it is minutes of pure latency, and
# you pay it again on every runtime restart. Copying the tree to the VM's local
# disk once costs ~30s and makes the loader cell roughly an order of magnitude
# faster. The copy is scratch -- it dies with the VM, and nothing writes back.
COLAB_CACHE_LOCAL = False

# --- Local only -------------------------------------------------------------
LOCAL_CANDIDATES = ("data", "encoder2/data")   # first one that exists wins

from pathlib import Path


def _detect_colab() -> bool:
    # importlib.util.find_spec("google.colab") is not usable here: it imports the
    # parent package first and raises outright when `google` does not exist.
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = _detect_colab() if RUNTIME == "auto" else RUNTIME == "colab"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT_ROOT = Path(DRIVE_DATA), Path(DRIVE_OUT)
    if not ROOT.exists():
        raise FileNotFoundError(
            f"{ROOT} not on your Drive -- set DRIVE_DATA to the folder holding user*/session_*")
    if COLAB_CACHE_LOCAL:
        import shutil, time as _time
        cache = Path("/content/trace-data/balabit")
        if not cache.exists():
            print(f"[colab] caching {ROOT} -> {cache} (once per VM) ...")
            _t0 = _time.time()
            shutil.copytree(ROOT, cache)
            print(f"[colab] copied in {_time.time() - _t0:.0f}s")
        ROOT = cache
else:
    # works whether the kernel starts in encoder2/ or at the repo root
    ROOT = next((p for p in map(Path, LOCAL_CANDIDATES) if p.is_dir()),
                Path(LOCAL_CANDIDATES[0])).resolve()
    OUT_ROOT = ROOT          # artifacts sit beside the data, and ./data is gitignored

OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"runtime : {'colab' if IN_COLAB else 'local'}")
print(f"data    : {ROOT}   (exists: {ROOT.exists()})")
print(f"outputs : {OUT_ROOT}")
print("users   :", sorted(p.name for p in ROOT.glob("user*") if p.is_dir()))

In [4]:
import pandas as pd

# Balabit session schema
# NOTE: timestamps are float64 on purpose. float32 holds ~7 significant digits;
# if either clock is a Unix epoch (~1.4e9) instead of session-relative seconds,
# float32 quantises it to ~100 s and every dt/velocity downstream is garbage.
DTYPES = {
    'record timestamp': 'float64',
    'client timestamp': 'float64',
    'button':           'category',
    'state':            'category',
    # float, not int: a single malformed or empty x/y cell makes an int dtype
    # raise and kills the whole read. float32 is exact for integers < 2**24.
    'x':                'float32',
    'y':                'float32',
}

def load_session(path):
    return pd.read_csv(path, dtype=DTYPES)

def user_key(p):
    # handles both 'user_7' (original Balabit layout) and 'user7' (./data layout)
    return int(''.join(c for c in p.name if c.isdigit()))

users = {}   # 'user7' -> {'session_0123456789': DataFrame}
for user_dir in sorted((p for p in ROOT.glob('user*') if p.is_dir()), key=user_key):
    sessions = {f.stem: load_session(f) for f in sorted(user_dir.glob('session_*'))}
    users[user_dir.name] = sessions
    print(f'{user_dir.name}: {len(sessions)} sessions, '
          f'{sum(len(d) for d in sessions.values()):,} events')

# flat list if you prefer positional access:
all_users = [list(users[u].values()) for u in users]        # list[list[DataFrame]]
all_sessions = [df for u in users.values() for df in u.values()]

ModuleNotFoundError: No module named 'pandas'

## Data preprocessing

raw event log -> per-stroke feature table.

Input CSV format (Balabit-style):
*    record timestamp, client timestamp,button,state,x,y

Pipeline:
*    load_events()      clean + dedupe + normalise the raw log
*    segment_strokes()  cut the move stream into strokes
*    strokes_to_frame() one row of ~55 features per stroke
*    strokes_to_windows()  (optional) aggregate N strokes -> one model input vector

Colab:
*    !pip -q install pandas numpy
*    from mouse_strokes import *
*    cfg = Config()
*    df  = process_file("session_0001", cfg, user="user7", session="session_0001")



In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------


@dataclass
class Config:
    # --- which clock to trust -------------------------------------------------
    time_col: str = "client timestamp"   # or "record timestamp"

    # --- stroke segmentation --------------------------------------------------
    pause_split_s: float = 0.50          # gap that ends a stroke
    adaptive_pause: bool = True          # override with k * median(dt) if larger
    adaptive_pause_mult: float = 4.0
    split_on_click: bool = True          # a click always terminates a stroke
    max_duration_s: float = 10.0         # hard cap so one stroke can't run away
    max_points: int = 300

    # --- stroke validity ------------------------------------------------------
    min_points: int = 6
    min_path_px: float = 20.0
    min_duration_s: float = 0.05

    # --- feature knobs --------------------------------------------------------
    dir_change_deg: float = 20.0         # angle change that counts as a reversal
    stationary_px: float = 2.0           # segment shorter than this == micro-pause
    tail_frac: float = 0.25              # "final approach" = last 25% of the stroke

    # --- normalisation --------------------------------------------------------
    screen_w: float | None = None        # e.g. 1920 -> positions/lengths in screen units
    screen_h: float | None = None

    # --- misc -----------------------------------------------------------------
    verbose: bool = True


MOVE_STATES = {"move", "drag"}
DOWN_STATES = {"pressed", "down"}
UP_STATES = {"released", "up"}


# ----------------------------------------------------------------------------
# small numeric helpers
# ----------------------------------------------------------------------------


def _safe_div(a, b, fill=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full(a.shape, fill, dtype=float)
    m = np.abs(b) > 1e-12
    out[m] = a[m] / b[m]
    return out


def _wrap(a):
    """Wrap angles to (-pi, pi]."""
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


def _ffill_invalid(values, valid):
    """Replace invalid entries with the last valid one (vectorised)."""
    values = np.asarray(values, dtype=float)
    valid = np.asarray(valid, dtype=bool)
    if not valid.any():
        return np.zeros_like(values)
    idx = np.where(valid, np.arange(len(values)), 0)
    idx = np.maximum.accumulate(idx)
    return values[idx]


def _stats(prefix, arr, pcts=(25, 50, 75)):
    """mean/std/min/max/percentiles of an array, as a flat dict."""
    out = {}
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        keys = ["mean", "std", "min", "max"] + [f"p{p}" for p in pcts]
        return {f"{prefix}_{k}": np.nan for k in keys}
    out[f"{prefix}_mean"] = float(arr.mean())
    out[f"{prefix}_std"] = float(arr.std(ddof=0))
    out[f"{prefix}_min"] = float(arr.min())
    out[f"{prefix}_max"] = float(arr.max())
    for p, q in zip(pcts, np.percentile(arr, pcts)):
        out[f"{prefix}_p{p}"] = float(q)
    return out


# ----------------------------------------------------------------------------
# 1. load
# ----------------------------------------------------------------------------


def load_events(path, cfg: Config = Config()) -> pd.DataFrame:
    """Read a raw log and return a clean frame with columns t, x, y, state, button, is_move."""
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    tcol = cfg.time_col.strip().lower()
    if tcol not in df.columns:
        raise KeyError(f"time column {tcol!r} not in {list(df.columns)}")
    for c in ("x", "y", "state"):
        if c not in df.columns:
            raise KeyError(f"expected column {c!r} in {list(df.columns)}")

    out = pd.DataFrame(
        {
            "t": pd.to_numeric(df[tcol], errors="coerce"),
            "x": pd.to_numeric(df["x"], errors="coerce"),
            "y": pd.to_numeric(df["y"], errors="coerce"),
            "state": df["state"].astype(str).str.strip().str.lower(),
            "button": df.get("button", "NoButton").astype(str).str.strip().str.lower(),
        }
    ).dropna(subset=["t", "x", "y"])

    # Balabit sometimes parks the cursor at absurd coordinates on session start/end
    out = out[(out.x.between(-1e4, 1e5)) & (out.y.between(-1e4, 1e5))]

    out = out.sort_values("t", kind="mergesort").reset_index(drop=True)
    out["is_move"] = out.state.isin(MOVE_STATES)

    # Ties: several move samples share one timestamp (event coalescing in the log).
    # Keep the LAST position of each tied group; never drop button events.
    moves = out[out.is_move].drop_duplicates(subset="t", keep="last")
    others = out[~out.is_move]
    out = (
        pd.concat([moves, others])
        .sort_values(["t"], kind="mergesort")
        .reset_index(drop=True)
    )

    if cfg.screen_w:
        out["x"] = out["x"] / cfg.screen_w
    if cfg.screen_h:
        out["y"] = out["y"] / cfg.screen_h

    return out


def diagnose(events: pd.DataFrame, cfg: Config = Config()) -> dict:
    """Sampling-rate report. Run this once per dataset before trusting any derivative."""
    mv = events[events.is_move]
    dt = np.diff(mv.t.values)
    dt = dt[dt > 0]
    d = {
        "n_events": len(events),
        "n_moves": len(mv),
        "n_clicks": int(events.state.isin(DOWN_STATES).sum()),
        "duration_s": float(events.t.max() - events.t.min()) if len(events) else 0.0,
        "dt_median": float(np.median(dt)) if dt.size else np.nan,
        "dt_p90": float(np.percentile(dt, 90)) if dt.size else np.nan,
        "sample_rate_hz": float(1.0 / np.median(dt)) if dt.size else np.nan,
    }
    if cfg.verbose:
        print(
            f"[diagnose] {d['n_moves']} moves, {d['n_clicks']} clicks, "
            f"{d['duration_s']:.0f}s, median dt={d['dt_median']*1000:.0f}ms "
            f"(~{d['sample_rate_hz']:.0f} Hz)"
        )
        if d["sample_rate_hz"] < 30:
            print(
                "  ! low sampling rate: jerk and spectral features will be mostly "
                "quantisation noise. Trust speed/geometry/timing features instead."
            )
        if cfg.pause_split_s < 3 * d["dt_median"]:
            print(
                f"  ! pause_split_s={cfg.pause_split_s}s is close to the sampling "
                f"interval; almost every sample would start a new stroke."
            )
    return d


# ----------------------------------------------------------------------------
# 2. click pairing
# ----------------------------------------------------------------------------


def pair_clicks(events: pd.DataFrame) -> pd.DataFrame:
    """Match each press to the next release of the same button -> dwell time."""
    ev = events[events.state.isin(DOWN_STATES | UP_STATES)]
    rows, open_press = [], {}
    for r in ev.itertuples():
        if r.state in DOWN_STATES:
            open_press[r.button] = r
        elif r.state in UP_STATES and r.button in open_press:
            p = open_press.pop(r.button)
            rows.append(
                {
                    "t_down": p.t,
                    "t_up": r.t,
                    "dwell": r.t - p.t,
                    "button": r.button,
                    "cx": p.x,
                    "cy": p.y,
                }
            )
    return pd.DataFrame(rows, columns=["t_down", "t_up", "dwell", "button", "cx", "cy"])


# ----------------------------------------------------------------------------
# 3. segmentation
# ----------------------------------------------------------------------------


def segment_strokes(events: pd.DataFrame, cfg: Config = Config()):
    """Cut the move stream into strokes. Returns a list of (t, x, y, is_drag, click_row_or_None)."""
    mv = events[events.is_move].reset_index(drop=True)
    if len(mv) < 2:
        return []

    thr = cfg.pause_split_s
    if cfg.adaptive_pause:
        dt_all = np.diff(mv.t.values)
        dt_all = dt_all[dt_all > 0]
        if dt_all.size:
            thr = max(thr, cfg.adaptive_pause_mult * float(np.median(dt_all)))

    t = mv.t.values.astype(float)
    x = mv.x.values.astype(float)
    y = mv.y.values.astype(float)
    is_drag = (mv.state == "drag").values

    # how many click events have happened before each move sample
    click_cum = events.state.isin(DOWN_STATES | UP_STATES).cumsum()
    click_cum_mv = click_cum[events.is_move.values].values

    dt = np.diff(t)
    brk = np.zeros(len(t), dtype=bool)
    brk[1:] |= dt > thr                       # pause
    brk[1:] |= dt <= 0                        # clock glitch
    brk[1:] |= is_drag[1:] != is_drag[:-1]    # drag <-> free move
    if cfg.split_on_click:
        brk[1:] |= np.diff(click_cum_mv) > 0  # a click happened in between

    sid = np.cumsum(brk)

    clicks = pair_clicks(events)
    strokes = []
    for _, idx in pd.Series(np.arange(len(t))).groupby(sid):
        idx = idx.values
        # hard caps: chop over-long strokes into pieces
        pieces = [idx]
        if len(idx) > cfg.max_points:
            pieces = [
                idx[i : i + cfg.max_points] for i in range(0, len(idx), cfg.max_points)
            ]
        for p in pieces:
            if len(p) < 2:
                continue
            tt, xx, yy = t[p], x[p], y[p]
            if tt[-1] - tt[0] > cfg.max_duration_s:
                keep = tt - tt[0] <= cfg.max_duration_s
                tt, xx, yy = tt[keep], xx[keep], yy[keep]
                if len(tt) < 2:
                    continue
            # click that terminates this stroke, if any (press within 1s after the end)
            click = None
            if len(clicks):
                cand = clicks[(clicks.t_down >= tt[-1] - 1e-9) & (clicks.t_down <= tt[-1] + 1.0)]
                if len(cand):
                    click = cand.iloc[0]
            strokes.append((tt, xx, yy, bool(is_drag[p[0]]), click))
    return strokes


# ----------------------------------------------------------------------------
# 4. per-stroke features
# ----------------------------------------------------------------------------


def stroke_features(t, x, y, is_drag=False, click=None, cfg: Config = Config()) -> dict:
    n = len(t)
    f: dict = {}

    dt = np.diff(t)
    dx, dy = np.diff(x), np.diff(y)
    seg = np.hypot(dx, dy)
    v = _safe_div(seg, dt)

    duration = float(t[-1] - t[0])
    path = float(seg.sum())
    disp = float(np.hypot(x[-1] - x[0], y[-1] - y[0]))

    # --- shape ---------------------------------------------------------------
    f["n_points"] = n
    f["duration"] = duration
    f["path_len"] = path
    f["displacement"] = disp
    f["straightness"] = disp / path if path > 0 else 0.0
    f["is_drag"] = int(is_drag)

    bw, bh = float(x.max() - x.min()), float(y.max() - y.min())
    f["bbox_w"], f["bbox_h"] = bw, bh
    f["bbox_area"] = bw * bh
    f["bbox_aspect"] = np.log1p(bw) - np.log1p(bh)          # symmetric, no div-by-zero
    f["angle_start_end"] = float(np.arctan2(y[-1] - y[0], x[-1] - x[0]))
    f["dir_sin"] = float(np.sin(f["angle_start_end"]))      # layout-invariant direction
    f["dir_cos"] = float(np.cos(f["angle_start_end"]))

    # deviation from the straight start->end line ("how bowed is the path")
    if disp > 1e-9:
        ux, uy = (x[-1] - x[0]) / disp, (y[-1] - y[0]) / disp
        perp = np.abs((x - x[0]) * (-uy) + (y - y[0]) * ux)
        f["dev_mean"] = float(perp.mean())
        f["dev_max"] = float(perp.max())
        f["dev_max_norm"] = float(perp.max() / disp)
    else:
        f["dev_mean"] = f["dev_max"] = f["dev_max_norm"] = 0.0

    # --- speed ---------------------------------------------------------------
    f.update(_stats("v", v))
    f["v_cv"] = f["v_std"] / f["v_mean"] if f["v_mean"] else np.nan
    tm = 0.5 * (t[1:] + t[:-1])                              # midpoint times for v
    if v.size:
        f["t_peak_frac"] = float((tm[int(np.argmax(v))] - t[0]) / duration) if duration > 0 else np.nan
        f["v_terminal"] = float(v[-1])
        f["v_initial"] = float(v[0])
    else:
        f["t_peak_frac"] = f["v_terminal"] = f["v_initial"] = np.nan

    # --- acceleration / jerk (weak at low sample rates -- see diagnose()) -----
    if n >= 3:
        a = _safe_div(np.diff(v), np.diff(tm))
        f.update(_stats("a", a))
        f["a_abs_mean"] = float(np.abs(a).mean())
        f["accel_frac"] = float((a > 0).mean())              # share of time speeding up
    else:
        f.update(_stats("a", np.array([])))
        f["a_abs_mean"] = f["accel_frac"] = np.nan

    if n >= 4:
        tj = 0.5 * (tm[1:] + tm[:-1])
        j = _safe_div(np.diff(a), np.diff(tj))
        f["j_abs_mean"] = float(np.abs(j).mean())
        f["j_abs_max"] = float(np.abs(j).max())
        f["j_std"] = float(j.std(ddof=0))
    else:
        f["j_abs_mean"] = f["j_abs_max"] = f["j_std"] = np.nan

    # --- angular ------------------------------------------------------------
    th = _ffill_invalid(np.arctan2(dy, dx), seg > 1e-12)
    if th.size >= 2:
        dth = _wrap(np.diff(th))
        angv = _safe_div(dth, np.diff(tm))
        curv = _safe_div(dth, 0.5 * (seg[1:] + seg[:-1]))
        f["angle_abs_sum"] = float(np.abs(dth).sum())
        f["angle_abs_mean"] = float(np.abs(dth).mean())
        f["angv_abs_mean"] = float(np.abs(angv).mean())
        f["angv_abs_max"] = float(np.abs(angv).max())
        f["curv_abs_mean"] = float(np.abs(curv).mean())
        f["curv_abs_max"] = float(np.abs(curv).max())
        f["curv_std"] = float(curv.std(ddof=0))
        nd = int((np.abs(dth) > np.deg2rad(cfg.dir_change_deg)).sum())
        f["n_dir_changes"] = nd
        f["dir_change_rate"] = nd / duration if duration > 0 else np.nan
    else:
        for k in (
            "angle_abs_sum angle_abs_mean angv_abs_mean angv_abs_max "
            "curv_abs_mean curv_abs_max curv_std n_dir_changes dir_change_rate"
        ).split():
            f[k] = np.nan

    # --- sampling / rhythm ---------------------------------------------------
    f["dt_mean"] = float(dt.mean())
    f["dt_std"] = float(dt.std(ddof=0))
    f["dt_cv"] = f["dt_std"] / f["dt_mean"] if f["dt_mean"] else np.nan
    micro = seg < cfg.stationary_px
    f["n_micro_pauses"] = int(micro.sum())
    f["micro_pause_frac"] = float(dt[micro].sum() / duration) if duration > 0 else np.nan

    # --- final approach (the part that discriminates most, per the literature) -
    tail = tm >= (t[-1] - cfg.tail_frac * duration) if duration > 0 else np.zeros_like(tm, bool)
    if tail.any():
        f["tail_v_mean"] = float(v[tail].mean())
        f["tail_v_max"] = float(v[tail].max())
        f["tail_path_frac"] = float(seg[tail].sum() / path) if path > 0 else np.nan
    else:
        f["tail_v_mean"] = f["tail_v_max"] = f["tail_path_frac"] = np.nan

    # --- click ---------------------------------------------------------------
    if click is not None:
        cx, cy = float(click.cx), float(click.cy)
        d = np.hypot(x - cx, y - cy)
        i = int(np.argmin(d))
        f["has_click"] = 1
        f["click_dwell"] = float(click.dwell)
        f["click_button_left"] = int(str(click.button).startswith("left"))
        f["click_delay"] = float(click.t_down - t[-1])       # move end -> press
        f["click_dist_end"] = float(np.hypot(x[-1] - cx, y[-1] - cy))
        f["overshoot_path"] = float(seg[i:].sum())           # path travelled after closest approach
        f["overshoot_max"] = float(d[i:].max())
    else:
        for k in (
            "has_click click_dwell click_button_left click_delay "
            "click_dist_end overshoot_path overshoot_max"
        ).split():
            f[k] = 0 if k == "has_click" else np.nan

    return f


# ----------------------------------------------------------------------------
# 5. file / directory drivers
# ----------------------------------------------------------------------------


def strokes_to_frame(strokes, cfg: Config = Config()) -> pd.DataFrame:
    rows, dropped = [], {"short": 0, "tiny_path": 0, "brief": 0}
    for k, (t, x, y, is_drag, click) in enumerate(strokes):
        if len(t) < cfg.min_points:
            dropped["short"] += 1
            continue
        if float(np.hypot(np.diff(x), np.diff(y)).sum()) < cfg.min_path_px:
            dropped["tiny_path"] += 1
            continue
        if t[-1] - t[0] < cfg.min_duration_s:
            dropped["brief"] += 1
            continue
        f = stroke_features(t, x, y, is_drag, click, cfg)
        f["stroke_id"] = k
        f["t_start"] = float(t[0])
        f["t_end"] = float(t[-1])
        rows.append(f)

    if cfg.verbose:
        print(
            f"[strokes] kept {len(rows)} / {len(strokes)} "
            f"(dropped: {dropped['short']} too few points, "
            f"{dropped['tiny_path']} too short, {dropped['brief']} too brief)"
        )
    df = pd.DataFrame(rows)
    return df.replace([np.inf, -np.inf], np.nan)


def process_file(path, cfg: Config = Config(), user=None, session=None) -> pd.DataFrame:
    ev = load_events(path, cfg)
    if cfg.verbose:
        diagnose(ev, cfg)
    df = strokes_to_frame(segment_strokes(ev, cfg), cfg)
    if len(df):
        df.insert(0, "session", session or Path(path).name)
        df.insert(0, "user", user or Path(path).parent.name)
    return df


def process_dir(root, cfg: Config = Config(), pattern="**/session_*") -> pd.DataFrame:
    """Walk a Balabit-style tree (root/userN/session_xxxx) into one long frame."""
    root = Path(root)
    out = []
    for p in sorted(root.glob(pattern)):
        if not p.is_file():
            continue
        try:
            d = process_file(p, cfg, user=p.parent.name, session=p.name)
            if len(d):
                out.append(d)
        except Exception as e:  # keep going -- some session files are truncated
            warnings.warn(f"{p}: {e}")
    if not out:
        return pd.DataFrame()
    df = pd.concat(out, ignore_index=True)
    if cfg.verbose:
        print(f"[process_dir] {len(df)} strokes from {df.user.nunique()} users")
    return df


# ----------------------------------------------------------------------------
# 6. optional: strokes -> model input windows
# ----------------------------------------------------------------------------


NON_FEATURE = {"user", "session", "stroke_id", "t_start", "t_end"}


def strokes_to_windows(
    df: pd.DataFrame,
    n: int = 25,
    stride: int = 5,
    min_strokes: int = 8,
    max_span_s: float = 60.0,
    aggs=("mean", "std", "p25", "p50", "p75"),
) -> pd.DataFrame:
    """Sliding window over strokes -> one wide vector per window.

    Windows spanning more than max_span_s are dropped (the user walked away).
    Windows with fewer than min_strokes are never emitted (idle reading).
    """
    feat_cols = [c for c in df.columns if c not in NON_FEATURE]
    rows = []
    for (u, s), g in df.groupby(["user", "session"], sort=False):
        g = g.sort_values("t_start").reset_index(drop=True)
        vals = g[feat_cols].to_numpy(dtype=float)
        for end in range(n, len(g) + 1, stride):
            start = end - n
            if end - start < min_strokes:
                continue
            span = g.t_end.iloc[end - 1] - g.t_start.iloc[start]
            if span > max_span_s:
                continue
            blk = vals[start:end]
            rec = {"user": u, "session": s,
                   "t_start": float(g.t_start.iloc[start]),
                   "t_end": float(g.t_end.iloc[end - 1]),
                   "n_strokes": end - start, "span_s": float(span)}
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                for agg in aggs:
                    if agg == "mean":
                        vv = np.nanmean(blk, axis=0)
                    elif agg == "std":
                        vv = np.nanstd(blk, axis=0)
                    else:
                        vv = np.nanpercentile(blk, int(agg[1:]), axis=0)
                    for c, val in zip(feat_cols, vv):
                        rec[f"{c}_{agg}"] = float(val)
            rows.append(rec)
    out = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    print(f"[windows] {len(out)} windows x {len([c for c in out.columns if c not in NON_FEATURE | {'n_strokes','span_s'}])} features")
    return out


# ----------------------------------------------------------------------------


import sys

if __name__ == "__main__" and "ipykernel" not in sys.modules:

    cfg = Config()
    src = Path(sys.argv[1]) if len(sys.argv) > 1 else None
    if src is None:
        print(__doc__)
    elif src.is_dir():
        d = process_dir(src, cfg)
        d.to_csv("strokes.csv", index=False)
    else:
        d = process_file(src, cfg)
        d.to_csv("strokes.csv", index=False)
        print(d.head())

---
## 1. Event-level cleaning

`load_events()` in the cell above already did the bare minimum (sort, drop ties,
clip absurd coordinates). This replaces it with the full clean, and makes the same
cleaning reachable from an **in-memory** DataFrame, which is what the ./data loader
gives us.

What gets removed and why:

| step | what it kills | why it matters |
|---|---|---|
| `dropna` | rows with unparseable `t/x/y` | one NaN poisons `np.diff` for the whole stroke |
| `drop_scroll` | wheel events | they carry `Down`/`Up` states and would be paired as fake clicks with fake dwell times |
| range clip | cursor parked at `(-1e6, …)` | Balabit does this at session start/end |
| out-of-order | rows where the clock steps backwards | sorting them into place is worse than dropping — it reorders real motion |
| tied timestamps | coalesced move samples | `dt = 0` → infinite velocity |
| despike | isolated 1-sample position jumps | remote-desktop capture artifact: out and immediately back. Detected as `d(i-1,i)` and `d(i,i+1)` both large while `d(i-1,i+1)` is small |
| `collapse_frozen` | repeated identical positions | **off by default** — a frozen cursor is real behaviour (the user stopped), and `micro_pause_frac` is built to measure it |

Not done on purpose: **no smoothing / Savitzky-Golay filter.** Any low-pass filter
destroys exactly the high-frequency content that `j_abs_mean`, `j_std` and `dt_cv`
are trying to measure. At ~60 Hz those features are already marginal; filtering
would make them look clean and mean nothing.

In [ ]:
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd, warnings, json, time

@dataclass
class PrepConfig:
    # --- event level ---
    x_range: tuple = (-2000.0, 8000.0)
    y_range: tuple = (-2000.0, 5000.0)
    drop_scroll: bool = True
    drop_out_of_order: bool = True
    despike: bool = True
    despike_px: float = 250.0
    despike_ratio: float = 0.35
    collapse_frozen: bool = False
    frozen_px: float = 0.0
    # --- session level ---
    min_events: int = 150
    min_moves: int = 100
    min_duration_s: float = 20.0
    max_dt_median_s: float = 0.40
    # --- stroke level ---
    max_speed_px_s: float = 6000.0
    max_path_px: float = 20000.0
    drop_zero_var_strokes: bool = True
    # --- feature level ---
    max_nan_frac: float = 0.50
    min_unique: int = 3
    winsor_mad: float = 6.0
    log1p_skew: float = 3.0
    scaler: str = "robust"          # "robust" | "standard" | "none"
    corr_prune: float = 0.0         # 0 disables; else drop |r| above this


def _despike_mask(x, y, jump_px, ratio):
    n = len(x)
    bad = np.zeros(n, dtype=bool)
    if n < 3:
        return bad
    d = np.hypot(np.diff(x), np.diff(y))
    a, b = d[:-1], d[1:]
    c = np.hypot(x[2:] - x[:-2], y[2:] - y[:-2])
    bad[1:-1] = (a > jump_px) & (b > jump_px) & (c < ratio * (a + b))
    return bad


def clean_events(raw: pd.DataFrame, cfg: Config = Config(), pcfg: PrepConfig = PrepConfig()):
    rep = {"n_raw": len(raw)}
    df = raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    tcol = cfg.time_col.strip().lower()
    for c in (tcol, "x", "y", "state"):
        if c not in df.columns:
            raise KeyError(f"expected column {c!r} in {list(df.columns)}")

    out = pd.DataFrame({
        "t":      pd.to_numeric(df[tcol], errors="coerce").astype("float64"),
        "x":      pd.to_numeric(df["x"], errors="coerce").astype("float64"),
        "y":      pd.to_numeric(df["y"], errors="coerce").astype("float64"),
        "state":  df["state"].astype(str).str.strip().str.lower(),
        "button": (df["button"] if "button" in df.columns else "nobutton").astype(str).str.strip().str.lower(),
    })

    n = len(out); out = out.dropna(subset=["t", "x", "y"]);            rep["drop_nan"] = n - len(out)

    if pcfg.drop_scroll:
        n = len(out)
        out = out[~(out.button.str.contains("scroll") | out.state.str.contains("scroll"))]
        rep["drop_scroll"] = n - len(out)

    n = len(out)
    out = out[out.x.between(*pcfg.x_range) & out.y.between(*pcfg.y_range)];  rep["drop_oob"] = n - len(out)

    if pcfg.drop_out_of_order and len(out):
        t = out.t.values
        keep = t >= np.maximum.accumulate(t) - 1e-9
        rep["drop_backwards"] = int((~keep).sum())
        out = out[keep]
    else:
        rep["drop_backwards"] = 0

    out = out.sort_values("t", kind="mergesort").reset_index(drop=True)
    out["is_move"] = out.state.isin(MOVE_STATES)

    n = len(out)
    moves  = out[out.is_move].drop_duplicates(subset="t", keep="last")
    others = out[~out.is_move]
    out = pd.concat([moves, others]).sort_values("t", kind="mergesort").reset_index(drop=True)
    rep["drop_tied_t"] = n - len(out)

    if pcfg.despike and out.is_move.any():
        mv = out.index[out.is_move.values].to_numpy()
        bad = _despike_mask(out.x.values[mv], out.y.values[mv], pcfg.despike_px, pcfg.despike_ratio)
        rep["drop_spikes"] = int(bad.sum())
        if bad.any():
            out = out.drop(index=mv[bad]).reset_index(drop=True)
    else:
        rep["drop_spikes"] = 0

    if pcfg.collapse_frozen and out.is_move.any():
        mv = out.is_move.values
        same = np.zeros(len(out), dtype=bool)
        same[1:] = mv[1:] & mv[:-1] & (np.abs(np.diff(out.x.values)) <= pcfg.frozen_px) \
                                    & (np.abs(np.diff(out.y.values)) <= pcfg.frozen_px)
        rep["drop_frozen"] = int(same.sum())
        out = out[~same].reset_index(drop=True)
    else:
        rep["drop_frozen"] = 0

    if cfg.screen_w: out["x"] = out["x"] / cfg.screen_w
    if cfg.screen_h: out["y"] = out["y"] / cfg.screen_h

    rep["n_clean"] = len(out)
    return out.reset_index(drop=True), rep


def session_report(ev: pd.DataFrame, pcfg: PrepConfig = PrepConfig()) -> dict:
    mv = ev[ev.is_move]
    dt = np.diff(mv.t.values); dt = dt[dt > 0]
    d = {
        "n_events":  len(ev),
        "n_moves":   len(mv),
        "n_clicks":  int(ev.state.isin(DOWN_STATES).sum()),
        "duration_s": float(ev.t.max() - ev.t.min()) if len(ev) else 0.0,
        "dt_median": float(np.median(dt)) if dt.size else np.nan,
        "hz":        float(1 / np.median(dt)) if dt.size else np.nan,
    }
    reasons = []
    if d["n_events"]   < pcfg.min_events:      reasons.append("few_events")
    if d["n_moves"]    < pcfg.min_moves:       reasons.append("few_moves")
    if d["duration_s"] < pcfg.min_duration_s:  reasons.append("short")
    if not np.isfinite(d["dt_median"]) or d["dt_median"] > pcfg.max_dt_median_s:
        reasons.append("slow_sampling")
    d["keep"] = len(reasons) == 0
    d["drop_reason"] = ",".join(reasons)
    return d

---
## 2. Connect the loader to the feature extractor

The ./data loader produced `users['user7']['session_123'] -> DataFrame`, but
`process_file()` takes a *path*. `process_frame()` is the missing link, and
`load_events` is rebound so the path-based entry points (`process_file`,
`process_dir`) clean identically — one code path, no drift.

Runs `clean_events → session QC → segment_strokes → stroke_features` over
everything loaded, and returns a per-session QC table alongside the stroke table
so dropped sessions are auditable rather than silent.

`process_frame` now also hands back the **raw point arrays** of every surviving
stroke, keyed `(user, session, stroke_id)`. Section 7.5 packs those into the
tensor the encoder actually consumes; the feature table stays as the QC
instrument and the baseline.

In [ ]:
PREP = PrepConfig()

def load_events(path, cfg: Config = Config()) -> pd.DataFrame:      # noqa: F811
    """Overrides the version in the feature cell so file- and memory-paths clean identically."""
    return clean_events(pd.read_csv(path), cfg, PREP)[0]


def process_frame(raw: pd.DataFrame, user: str, session: str,
                  cfg: Config = Config(), pcfg: PrepConfig = PREP, keep_seqs: bool = True):
    """Returns (stroke feature table, QC record, raw point sequences).

    The feature table is no longer model input -- the encoder eats the raw
    sequences. It is kept because every QC threshold downstream (`v_max`,
    `path_len`, `duration`) is defined on it, and because it is the baseline the
    raw model has to beat.
    """
    ev, rep = clean_events(raw, cfg, pcfg)
    qc = {"user": user, "session": session, **rep, **session_report(ev, pcfg)}
    if not qc["keep"]:
        return pd.DataFrame(), qc, {}
    raw_strokes = segment_strokes(ev, cfg)
    st = strokes_to_frame(raw_strokes, cfg)
    qc["n_strokes"] = len(st)
    seqs = {}
    if len(st):
        st.insert(0, "session", session)
        st.insert(0, "user", user)
        if keep_seqs:
            # stroke_id indexes raw_strokes, so surviving strokes map straight back
            for sid in st.stroke_id.to_numpy():
                t, x, y, is_drag, _click = raw_strokes[int(sid)]
                seqs[(user, session, int(sid))] = (
                    np.asarray(t, dtype=np.float64),
                    np.asarray(x, dtype=np.float32),
                    np.asarray(y, dtype=np.float32),
                    bool(is_drag),
                )
    else:
        qc["keep"], qc["drop_reason"] = False, "no_strokes"
    return st, qc, seqs


def process_loaded(users: dict, cfg: Config = Config(), pcfg: PrepConfig = PREP,
                   progress_every: int = 100, keep_seqs: bool = True):
    cfg = replace(cfg, verbose=False)
    frames, qcs, seqs, t0, i = [], [], {}, time.time(), 0
    total = sum(len(s) for s in users.values())
    for u, sessions in users.items():
        for sname, raw in sessions.items():
            i += 1
            try:
                st, qc, sq = process_frame(raw, u, sname, cfg, pcfg, keep_seqs)
            except Exception as e:
                qcs.append({"user": u, "session": sname, "keep": False,
                            "drop_reason": f"error:{type(e).__name__}"})
                warnings.warn(f"{u}/{sname}: {e}")
                continue
            qcs.append(qc)
            if len(st):
                frames.append(st); seqs.update(sq)
            if progress_every and i % progress_every == 0:
                print(f"  {i}/{total} sessions  ({time.time()-t0:.0f}s)")
    qc = pd.DataFrame(qcs)
    strokes = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    n_pts = sum(len(v[0]) for v in seqs.values())
    print(f"[pipeline] {qc.keep.sum()}/{len(qc)} sessions kept, "
          f"{len(strokes):,} strokes, {n_pts:,} raw points, {time.time()-t0:.0f}s")
    if (~qc.keep).any():
        print(qc.loc[~qc.keep, "drop_reason"].value_counts().to_string())
    return strokes, qc, seqs


# --- run it -----------------------------------------------------------------
cfg  = Config(time_col="client timestamp", verbose=False)
PREP = PrepConfig()

strokes_raw, qc, SEQS = process_loaded(users, cfg, PREP, progress_every=100)
display(qc.head())
display(strokes_raw.head())

---
## 3. Stroke-table QC

Session-level QC kept or killed whole files; this drops individual bad strokes
that survived segmentation. `max_speed_px_s = 6000` is the operative one — real
sustained cursor motion tops out around 3000–4000 px/s, so anything above that
is a residual teleport the despiker missed.

**The tradeoff worth being explicit about:** this is a biometrics dataset, and an
"outlier" here is not necessarily noise — a user who flings the mouse is *supposed*
to look extreme. Every threshold below trades away some real between-user variance
to remove capture artifacts. Keep them loose (kill physics violations, not unusual
humans) and check the per-user drop rate: if one user loses far more strokes than
the rest, the filter is eating signal, not noise.

In [ ]:
def filter_strokes(df: pd.DataFrame, pcfg: PrepConfig = PREP) -> pd.DataFrame:
    n0 = len(df); drops = {}
    m = pd.Series(True, index=df.index)

    bad = df.v_max > pcfg.max_speed_px_s;        drops["impossible_speed"] = int(bad.sum()); m &= ~bad
    bad = df.path_len > pcfg.max_path_px;        drops["huge_path"] = int(bad.sum());        m &= ~bad
    if pcfg.drop_zero_var_strokes:
        bad = (df.v_std == 0) | (df.bbox_area == 0)
        drops["degenerate"] = int(bad.sum()); m &= ~bad
    bad = ~np.isfinite(df.duration) | (df.duration <= 0)
    drops["bad_duration"] = int(bad.sum()); m &= ~bad

    out = df[m].reset_index(drop=True)
    print(f"[strokes] {len(out):,}/{n0:,} kept  " +
          "  ".join(f"-{k}:{v}" for k, v in drops.items() if v))
    return out


strokes = filter_strokes(strokes_raw, PREP)

# per-user drop rate -- should be roughly flat across users
before = strokes_raw.groupby("user").size()
after  = strokes.groupby("user").size()
display(pd.DataFrame({"raw": before, "kept": after,
                      "drop_%": (100 * (1 - after / before)).round(1)}))

---
## 4. Session-level train/test split

Do this **before** fitting any scaler or imputer.

Two leaks to avoid, both of which will hand you 99% accuracy that collapses on real
data:

1. **Row-level splitting.** Consecutive strokes in one session are nearly
   duplicates. A random row split puts near-copies on both sides, so the model
   memorises sessions rather than users. Split by session.
2. **Fitting the preprocessor on everything.** Medians, MAD bounds and scale
   factors computed over train+test leak test distribution into training.
   Fit on train, transform test.

Balabit's own protocol goes further: train on a user's sessions, test on *held-out
sessions*, and for the impostor task test on sessions from users never seen in
training. If that's your task, split by user, not by session.

In [ ]:
def session_split(df: pd.DataFrame, test_frac: float = 0.3, seed: int = 0):
    rng = np.random.default_rng(seed)
    tr, te, notes = [], [], []
    for u, g in df.groupby("user", sort=True):
        sess = np.array(sorted(g.session.unique()))
        if len(sess) < 2:
            # only one session for this user: fall back to a chronological split
            gg = g.sort_values("t_start")
            cut = int(len(gg) * (1 - test_frac))
            tr.append(gg.iloc[:cut]); te.append(gg.iloc[cut:])
            notes.append(f"{u}: 1 session -> time split (within-session leakage possible)")
            continue
        rng.shuffle(sess)
        k = max(1, int(round(len(sess) * test_frac)))
        te_s = set(sess[:k])
        tr.append(g[~g.session.isin(te_s)]); te.append(g[g.session.isin(te_s)])
    train = pd.concat(tr, ignore_index=True)
    test  = pd.concat(te, ignore_index=True)
    for n in notes: print("  !", n)
    print(f"[split] train {len(train):,} strokes / {train.session.nunique()} sessions | "
          f"test {len(test):,} / {test.session.nunique()} sessions | "
          f"users {train.user.nunique()}")
    assert set(train.session) & set(test.session) == set() or notes, "session leak"
    return train, test


train_s, test_s = session_split(strokes, test_frac=0.30, seed=0)

**félkövér szöveg**---
## 5. Fit preprocessing on train, apply to both

Order is: drop dead columns → `log1p` heavy tails → winsorize → impute → scale.

- **Winsorize, don't drop.** Clipping at median ± 6·MAD keeps the row and its label;
  dropping rows throws away strokes and biases the class balance toward whoever
  happens to be tidy. MAD is used rather than std because std is itself dragged
  around by the outliers it is meant to detect.
- **`log1p` before clipping.** `path_len`, `j_abs_max`, `curv_abs_max` etc. are
  log-normal-ish; on the raw scale a 6·MAD bound clips the top decile of a
  perfectly ordinary distribution.
- **Structural NaNs are not missing data.** `click_dwell` is NaN for strokes with
  no click — roughly a third of them. Median-imputing fabricates a dwell time for
  a click that never happened. Those columns fill with 0 and `has_click` carries
  the information; everything else fills with the train median.
- **`scaler="robust"`** (median / IQR) by default. Irrelevant for trees, essential
  for SVM/kNN/NN.
- **`corr_prune`** is off by default. It helps linear models and hurts nothing for
  trees, but it silently decides which of two correlated features to keep.

In [ ]:
NON_FEATURE_ALL = {"user", "session", "stroke_id", "t_start", "t_end", "n_strokes", "span_s"}
# structurally-missing columns: NaN means "no click happened", not "unknown"
STRUCTURAL_NAN = {"click_dwell", "click_button_left", "click_delay", "click_dist_end",
                  "overshoot_path", "overshoot_max"}


def fit_preprocessor(train: pd.DataFrame, pcfg: PrepConfig = PREP) -> dict:
    feats = [c for c in train.columns if c not in NON_FEATURE_ALL]
    X = train[feats].astype("float64")

    keep = [c for c in feats
            if X[c].isna().mean() <= pcfg.max_nan_frac
            and X[c].nunique(dropna=True) >= pcfg.min_unique]
    dropped_cols = sorted(set(feats) - set(keep))
    X = X[keep]

    # heavy right tails -> log1p (only strictly non-negative columns)
    logs = [c for c in keep
            if (X[c].dropna() >= 0).all() and abs(X[c].skew(skipna=True)) > pcfg.log1p_skew]
    X[logs] = np.log1p(X[logs])

    # robust bounds from median / MAD, percentile fallback when MAD collapses
    med = X.median()
    mad = (X - med).abs().median() * 1.4826
    lo = med - pcfg.winsor_mad * mad
    hi = med + pcfg.winsor_mad * mad
    flat = mad <= 1e-12
    if flat.any():
        lo[flat] = X.loc[:, flat].quantile(0.005)
        hi[flat] = X.loc[:, flat].quantile(0.995)
    X = X.clip(lo, hi, axis=1)

    fill = {c: (0.0 if c in STRUCTURAL_NAN else float(med[c])) for c in keep}

    if pcfg.scaler == "robust":
        center = X.median()
        q1, q3 = X.quantile(0.25), X.quantile(0.75)
        scale = (q3 - q1).replace(0, np.nan).fillna(X.std(ddof=0)).replace(0, 1.0)
    elif pcfg.scaler == "standard":
        center, scale = X.mean(), X.std(ddof=0).replace(0, 1.0)
    else:
        center = pd.Series(0.0, index=keep); scale = pd.Series(1.0, index=keep)

    pre = {"features": keep, "dropped_cols": dropped_cols, "log_cols": logs,
           "lo": lo.to_dict(), "hi": hi.to_dict(), "fill": fill,
           "center": center.to_dict(), "scale": scale.to_dict(),
           "config": asdict(pcfg)}

    if pcfg.corr_prune:
        Z = apply_preprocessor(train, pre)
        cm = Z.corr().abs().to_numpy(copy=True)
        np.fill_diagonal(cm, 0.0)
        drop, cols = set(), list(Z.columns)
        for i in range(len(cols)):
            if cols[i] in drop: continue
            for j in range(i + 1, len(cols)):
                if cm[i, j] > pcfg.corr_prune: drop.add(cols[j])
        pre["features"] = [c for c in keep if c not in drop]
        pre["corr_pruned"] = sorted(drop)
        print(f"[prep] corr>|{pcfg.corr_prune}| pruned {len(drop)} columns")

    print(f"[prep] {len(pre['features'])} features "
          f"(dropped {len(dropped_cols)} constant/empty, log1p on {len(logs)})")
    return pre


def apply_preprocessor(df: pd.DataFrame, pre: dict) -> pd.DataFrame:
    """Returns FEATURES ONLY, row order preserved -- no user/session/t_start columns.

    Keeping metadata out of the matrix is deliberate: `t_start` is a position in the
    session, and a model handed that column will happily learn it.
    """
    cols = pre["features"]
    X = df.reindex(columns=cols).astype("float64")
    logc = [c for c in pre["log_cols"] if c in cols]
    if logc:
        X[logc] = np.log1p(X[logc].clip(lower=-0.999999))
    X = X.clip(pd.Series(pre["lo"])[cols], pd.Series(pre["hi"])[cols], axis=1)
    X = X.fillna(pd.Series(pre["fill"])[cols])
    X = (X - pd.Series(pre["center"])[cols]) / pd.Series(pre["scale"])[cols]
    return X.replace([np.inf, -np.inf], 0.0).reset_index(drop=True)


pre     = fit_preprocessor(train_s, PREP)
X_train = apply_preprocessor(train_s, pre)      # features only
X_test  = apply_preprocessor(test_s,  pre)

y_train, y_test = train_s["user"].to_numpy(), test_s["user"].to_numpy()
g_train, g_test = train_s["session"].to_numpy(), test_s["session"].to_numpy()   # for GroupKFold
print(X_train.shape, X_test.shape)
display(X_train.describe().loc[["mean", "std", "min", "max"]].T.head(10))

---
## 6. Optional: stroke windows

One stroke is a weak signal; most of the Balabit literature aggregates. Build
windows from the **unscaled** stroke table, then split and scale the windows the
same way (aggregating already-scaled features would mean the window stats were
computed on a distribution fitted to individual strokes).

`max_span_s` guards against a window of 25 strokes that spans ten minutes because
the user walked away.

One quirk in `strokes_to_windows` as written: windows are always exactly `n`
strokes (`start = end - n`), so the `min_strokes` argument never fires and the
trailing partial window of each session is dropped. Harmless, but don't expect
`min_strokes=8` to do anything.

In [ ]:
windows = strokes_to_windows(strokes, n=25, stride=5, min_strokes=8, max_span_s=60.0)

if len(windows):
    train_w, test_w = session_split(windows, test_frac=0.30, seed=0)
    pre_w    = fit_preprocessor(train_w, PREP)
    Xw_train = apply_preprocessor(train_w, pre_w)
    Xw_test  = apply_preprocessor(test_w,  pre_w)
    yw_train, yw_test = train_w["user"].to_numpy(), test_w["user"].to_numpy()
    print(Xw_train.shape, Xw_test.shape)

---
## 7. Save

Parquet, not CSV: preserves dtypes, ~5–10× smaller, and loads in seconds instead of
re-walking 1700 CSVs off disk.

The `preprocessor.joblib` is the important artifact — it holds the clip bounds, fill
values and scale factors **fitted on train**. Any future session must go through the
same object, or inference silently sees a different feature space than training did.
The manifest records both configs so a result can be traced back to the settings that
produced it; bump `tag` when you change a threshold instead of overwriting.

In [ ]:
import joblib

def save_dataset(outdir, *, strokes=None, windows=None, train=None, test=None,
                 pre=None, qc=None, cfg=None, pcfg=None, tag="v1"):
    out = Path(outdir) / tag
    out.mkdir(parents=True, exist_ok=True)
    written = {}

    def _pq(name, df):
        if df is None or not len(df): return
        p = out / f"{name}.parquet"
        d = df.copy()
        for c in ("user", "session"):
            if c in d.columns: d[c] = d[c].astype(str)
        d.to_parquet(p, index=False, compression="zstd")
        written[name] = (str(p), d.shape, p.stat().st_size / 1e6)

    _pq("strokes", strokes); _pq("windows", windows)
    _pq("train", train);     _pq("test", test); _pq("session_qc", qc)

    if pre is not None:
        joblib.dump(pre, out / "preprocessor.joblib")
        (out / "preprocessor.json").write_text(json.dumps(pre, indent=2, default=float))
        written["preprocessor"] = (str(out / "preprocessor.joblib"), None, None)

    manifest = {
        "tag": tag,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "config": asdict(cfg) if cfg is not None else None,
        "prep_config": asdict(pcfg) if pcfg is not None else None,
        "shapes": {k: v[1] for k, v in written.items() if v[1]},
        "n_features": len(pre["features"]) if pre else None,
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

    print(f"[save] -> {out}")
    for k, (p, shape, mb) in written.items():
        print(f"   {k:12s} {str(shape):16s} {f'{mb:.1f} MB' if mb else ''}")
    return out


def load_dataset(outdir, tag="v1"):
    out = Path(outdir) / tag
    got = {p.stem: pd.read_parquet(p) for p in out.glob("*.parquet")}
    pj = out / "preprocessor.joblib"
    if pj.exists(): got["pre"] = joblib.load(pj)
    print(f"[load] {out}: " + ", ".join(f"{k}{getattr(v,'shape','')}" for k, v in got.items()))
    return got


SAVE_ROOT = OUT_ROOT / "processed"      # ./data/processed  |  Drive/trace-data/processed

save_dataset(
    SAVE_ROOT, tag="v1",
    strokes=strokes,
    windows=windows if "windows" in dir() and len(windows) else None,
    train=X_train.assign(user=y_train, session=g_train),
    test=X_test.assign(user=y_test, session=g_test),
    pre=pre, qc=qc, cfg=cfg, pcfg=PREP,
)

# later sessions start here instead of re-reading 1700 CSVs:
# d   = load_dataset(SAVE_ROOT, "v1")
# pre = d["pre"]; train = d["train"]; test = d["test"]

---
## 7.5 Raw stroke sequences

Everything above compresses a stroke into ~55 numbers that *I* chose. From here
the model input is the points themselves, and `stroke_features()` is demoted to
what it is good at: quality control and a baseline to beat.

Two properties of the capture make "just feed it the positions" harder than it
sounds, and handling them is the whole design:

| problem | usual fix | what is done here | why |
|---|---|---|---|
| the interval between samples varies — nominally ~60 Hz, but it drifts, drops frames, and coalesces | resample onto a uniform time grid | hand `dt` to the network as an input channel | interpolation invents points that were never observed, and the timing jitter it smooths away is itself identifying |
| strokes have different numbers of points | crop to a fixed length | pad to `max_len` + carry an explicit mask | every reduction in the encoder divides by the *true* length, so padding contributes nothing instead of dragging every mean toward zero |
| a few strokes are longer than `max_len` | crop the tail | thin uniformly and let `dt` absorb the wider spacing | cropping discards the end of the movement, which is exactly where the deceleration-onto-target behaviour lives |

`max_len` defaults to **64**, not 128. Balabit's capture is ~9 Hz for 87% of
sessions (only 211 of 1,676 run at 62.5 Hz), so the median stroke is 12 points and
p95 is 55: at 128 roughly 90% of every batch was padding, for 0.4% of strokes
resampled. Check the `[bank]` line — if your capture is faster, raise it.

**Channels per point** — all in physical units, none of them a statistic:

`dx`, `dy` (px since the previous sample) · `dt` (s since the previous sample) ·
`x_rel`, `y_rel` (px since the stroke started) · `x_abs`, `y_abs` (screen px) ·
`is_drag`

`x_rel`/`y_rel` are just `dx`/`dy` integrated, and `x_abs` is that plus an
offset — all three are redundant in the information-theoretic sense. They are
there because a 5-tap convolution reaches 5 samples, and handing it the running
integral saves it from having to learn one.

Velocity is deliberately **absent**. `dx/dt` is a function of three channels the
network already has, and hard-coding the division is the first step back down the
road this is trying to leave. If you want it back it is one line in
`stroke_channels()` — it is a reasonable thing to A/B.

**Where the scaling happens.** The bank stores raw pixels and seconds; centring
and scaling are applied at batch time from a scaler fitted on the training fold's
points only. Re-fitting the scaler therefore never means rebuilding the bank, and
the leakage discipline is the same one section 5 argues for.

In [ ]:
# ---------------------------------------------------------------------------
# Raw stroke sequences -- the tensor the encoder actually sees
# ---------------------------------------------------------------------------


@dataclass
class SeqConfig:
    max_len: int = 64                   # points kept per stroke (pad below, resample above)
                                        # 64 not 128: at Balabit's ~9 Hz the median
                                        # stroke is 12 points and p95 is 55, so 128
                                        # was ~90% padding for no gain
    resample_long: str = "uniform"      # "uniform" | "truncate"  -- only for n > max_len
    dt_clip_s: float = 1.0              # a gap longer than this is a pause, not a sample interval
    rate_norm_hz: float | None = None   # decimate every stroke toward this rate (see below)
    store_float16: bool = True          # halves the bank; coordinates are integers, so exact
    channels: tuple = ("dx", "dy", "dt", "x_rel", "y_rel", "x_abs", "y_abs", "is_drag")


# Per-channel centring / warping, all fitted on the TRAIN fold only.
#   centre "zero"   -- leave zero alone: a zero displacement must stay zero, and
#                      shifting dx/dy by a median would break the sign symmetry
#                      of "left" vs "right".
#   centre "median" -- absolute screen position, which has no meaningful origin.
#   warp   "log1p"  -- dt spans 4 orders of magnitude; linear scaling would make
#                      every ordinary 8 ms interval indistinguishable from zero.
#   warp   "unit"   -- already 0/1, never rescaled.
CHANNEL_SPEC = {
    "dx":      ("zero",   None),
    "dy":      ("zero",   None),
    "dt":      ("median", "log1p"),
    "x_rel":   ("zero",   None),
    "y_rel":   ("zero",   None),
    "x_abs":   ("median", None),
    "y_abs":   ("median", None),
    "is_drag": ("zero",   "unit"),
}


def stroke_channels(t, x, y, is_drag, scfg: SeqConfig):
    """One stroke -> (n, C) float32 in physical units (pixels, seconds).

    Nothing here is a statistic. dx/dy/dt are the raw increments between
    consecutive samples; x_rel/y_rel are those increments integrated (the shape
    of the path); x_abs/y_abs are where on the screen it happened. The encoder
    is left to work out what any of it means.
    """
    t = np.asarray(t, dtype=np.float64)
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n = len(t)

    # --- rate equalisation ---------------------------------------------------
    # In Balabit the capture rate tracks the user (user_0 is 54% 63 Hz sessions,
    # most others are ~2%), so `dt` is partly a machine fingerprint the encoder
    # can read instead of behaviour. Decimating each stroke toward a common rate
    # removes that shortcut.
    #
    # Deliberately NOT random augmentation. The median stroke here is 12 points;
    # a random factor of 2 leaves 6 and 3 leaves 4, so blanket decimation would
    # destroy the 87% of strokes that are already at 9 Hz in order to fix the 13%
    # that are not. An integer factor derived from each stroke's OWN spacing
    # leaves slow strokes untouched and only thins the fast ones.
    if scfg.rate_norm_hz and n > 2:
        dtm = float(np.median(np.diff(t)))
        if dtm > 0:
            f = int(round((1.0 / scfg.rate_norm_hz) / dtm))
            if f > 1 and n // f >= 6:               # never decimate below min_points
                t, x, y = t[::f], x[::f], y[::f]
                n = len(t)

    if n > scfg.max_len:
        if scfg.resample_long == "uniform":
            # keep the whole stroke, thin it out; dt absorbs the wider spacing so
            # the timing stays physically true
            sel = np.unique(np.linspace(0, n - 1, scfg.max_len).round().astype(int))
        else:
            sel = np.arange(scfg.max_len)
        t, x, y = t[sel], x[sel], y[sel]
        n = len(t)

    dt = np.diff(t, prepend=t[0])          # dt[0] = 0: "no previous sample"
    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])
    np.clip(dt, 0.0, scfg.dt_clip_s, out=dt)

    cols = {
        "dx": dx, "dy": dy, "dt": dt,
        "x_rel": x - x[0], "y_rel": y - y[0],
        "x_abs": x, "y_abs": y,
        "is_drag": np.full(n, float(is_drag)),
    }
    missing = set(scfg.channels) - set(cols)
    if missing:
        raise KeyError(f"unknown channel(s) {sorted(missing)}")
    return np.stack([cols[c] for c in scfg.channels], axis=1).astype(np.float32), n


def build_stroke_bank(df: pd.DataFrame, seqs: dict, scfg: SeqConfig = SeqConfig()):
    """Stroke table -> (S, lens, df) where S[i] is row i's padded raw sequence.

    Returns the table back because rows whose points went missing are dropped,
    and S is positional: row i of the returned frame is S[i] for the rest of the
    pipeline.
    """
    df = df.reset_index(drop=True)
    L, C = scfg.max_len, len(scfg.channels)
    store = np.float16 if scfg.store_float16 else np.float32
    S = np.zeros((len(df), L, C), dtype=store)        # allocated at final dtype: no float32 peak
    lens = np.zeros(len(df), dtype=np.int32)
    raw_n = np.zeros(len(df), dtype=np.int32)

    keys = zip(df.user.to_numpy(), df.session.to_numpy(), df.stroke_id.to_numpy())
    for i, (u, s, sid) in enumerate(keys):
        rec = seqs.get((str(u), str(s), int(sid)))       # numpy scalars -> plain keys
        if rec is None:
            continue
        t, x, y, drag = rec
        ch, n = stroke_channels(t, x, y, drag, scfg)
        S[i, :n] = ch
        lens[i], raw_n[i] = n, len(t)

    ok = lens > 0
    if not ok.all():
        print(f"[bank] {(~ok).sum()} strokes had no stored points -- dropped")
        S, lens, raw_n, df = S[ok], lens[ok], raw_n[ok], df[ok].reset_index(drop=True)

    trunc = float((raw_n > scfg.max_len).mean())
    if scfg.rate_norm_hz:
        thinned = float((lens < np.minimum(raw_n, scfg.max_len)).mean())
        print(f"[bank] rate-equalised to ~{scfg.rate_norm_hz:g} Hz: {thinned:.1%} of strokes thinned")
    print(f"[bank] {S.shape[0]:,} strokes x {L} x {C}ch  "
          f"({S.nbytes/1e6:.0f} MB {S.dtype})  "
          f"len median {int(np.median(lens))}, p95 {int(np.percentile(lens, 95))}, "
          f"{trunc:.1%} resampled down from >{L}")
    return S, lens, df


def fit_seq_scaler(S, lens, rows, scfg: SeqConfig = SeqConfig(),
                   max_strokes=20_000, max_points=2_000_000, seed=0):
    """Per-channel robust centre/scale from TRAIN points only.

    Fitted on points, not on strokes, so a long stroke contributes more samples
    than a short one -- which is what the encoder sees too.
    """
    rng = np.random.default_rng(seed)
    rows = np.asarray(rows)
    if len(rows) > max_strokes:
        rows = rng.choice(rows, max_strokes, replace=False)
    P = np.concatenate([S[i, :lens[i]] for i in rows]).astype(np.float32)
    if len(P) > max_points:
        P = P[rng.choice(len(P), max_points, replace=False)]

    C = len(scfg.channels)
    center = np.zeros(C, np.float32)
    scale = np.ones(C, np.float32)
    logm = np.zeros(C, bool)

    for j, c in enumerate(scfg.channels):
        how, warp = CHANNEL_SPEC[c]
        if warp == "unit":
            continue
        v = P[:, j]
        if warp == "log1p":
            v = np.log1p(np.clip(v, 0.0, None))
            logm[j] = True
        if how == "median":
            center[j] = float(np.median(v))
        s = 1.4826 * float(np.median(np.abs(v - center[j])))       # MAD, house style
        if s < 1e-9:
            q1, q3 = np.percentile(v, [25, 75]); s = float(q3 - q1)
        if s < 1e-9:
            s = float(v.std())
        scale[j] = s if s > 1e-9 else 1.0

    sc = {"channels": list(scfg.channels), "center": center, "scale": scale,
          "log": logm, "config": asdict(scfg), "n_points": int(len(P))}
    print("[scaler] fitted on {:,} points from {:,} strokes".format(len(P), len(rows)))
    for j, c in enumerate(scfg.channels):
        print(f"   {c:8s} centre {center[j]:10.3f}  scale {scale[j]:10.3f}"
              + ("  (log1p)" if logm[j] else ""))
    return sc

---
# 8. Siamese embedding model

Goal: a function `window of mouse activity -> vector`, trained so that
`cosine(z_a, z_b)` is high when two windows come from the same person and low
otherwise. Once that holds, everything else is arithmetic on vectors — enrol a
user by averaging their embeddings, verify a session by comparing against that
average, flag an intruder when similarity drops below a threshold.

**What is actually being embedded.** Not a user, and not a single stroke. A
*window* of 25 consecutive strokes from one session. One stroke is far too weak —
one mouse movement from two different people looks nearly identical. The *user*
embedding is a derived object: the mean of that user's window embeddings over
their enrolment sessions (`templates()` below). Keeping the model at window level
means new users can be enrolled without retraining, which is the whole point of
the metric-learning setup over a plain 10-way classifier.

**Two levels, and the split is the point.** Inside a stroke, time order is real
signal and a dilated conv stack reads it. Across the 25 strokes of a window,
order is close to arbitrary, and the set encoder throws it away.

**Architecture choices, and why:**

| choice | what I did | why |
|---|---|---|
| input | raw points, `(25, 128, 8)` per window + a length mask | no hand-chosen feature can cap what the model is allowed to notice. Irregular sampling is carried by the `dt` channel rather than resampled away |
| stroke encoder | masked dilated 1-D CNN → mean+std+max+attention over time → 128-d | convolution because the informative structure (acceleration onto a target, correction wobble) is local in time; dilation to reach ~29 samples without depth; every pool masked so length does not bias it |
| window encoder | per-stroke MLP → pool over the stroke axis → 64-d, L2-normalised | unchanged from the feature version — only what feeds it changed |
| pooling | attention + mean + std, concatenated | **permutation invariant on purpose.** Stroke order within a window is close to arbitrary, and a GRU/transformer over it mostly learns session-specific sequencing, which is exactly what fails to transfer to a new session. The `std` branch matters — *consistency* is as identifying as average speed |
| loss | batch-hard triplet with online mining | the modern form of the siamese idea. Random pairs are ~95% trivially-easy after a few epochs and the gradient dies. `loss="contrastive"` gives the classic pair loss if you want the comparison |
| batching | P users × K windows, K spread across distinct sessions | without this a random batch rarely contains a positive pair and there is nothing to mine. Spreading K over sessions makes the hardest positive a *cross-session* pair, which is the hard case that matters |
| output | L2-normalised, cosine similarity | fixes the scale so one threshold works across users |

**What this costs.** The conv front end runs on `P*K*25 = 1200` sequences per
step instead of 1200 short vectors, so a step is orders of magnitude more work
than the feature version was. Two consequences worth knowing before you hit run:
per-epoch validation is computed on a fixed subsample (`eval_max_windows`), and
`DEVICE` now prefers Apple-silicon `mps` when there is no CUDA. The full
evaluation in 8.4 still uses every window.

**The honest caveat, inverted from last time.** The model can now discover cues
`stroke_features()` never encoded — but with 10 users it can also memorise
capture quirks that correlate with a user's machine (screen size shows up in
`x_abs`, sampling rate in `dt`). The unseen-user block in 8.4 is the check that
matters; if seen-user numbers are excellent and unseen-user numbers are not, that
gap is what it is measuring. Dropping `x_abs`/`y_abs` from `SeqConfig.channels`
is the first thing to try.

In [ ]:
# Colab already has torch; uncomment if the runtime is bare.
# !pip -q install torch

import json, math, time, warnings
import joblib                     # save_model in 8.5; the raw path must not need the feature cells
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"                # Apple silicon; set DEVICE="cpu" if an op is unsupported
else:
    DEVICE = "cpu"
NON_FEATURE_ALL |= {"fold"}      # so the fold tag never becomes a feature
print("device:", DEVICE)


@dataclass
class ModelConfig:
    # --- windowing ---
    n_strokes: int = 12            # strokes per window (the unit that gets embedded)
    stride: int = 4
    max_span_s: float = 240.0      # drop windows where the user walked away mid-way
    # 12/4/240, not 25/8/90: at ~9 Hz a 25-stroke window routinely spans minutes, so
    # max_span_s=90 was silently discarding ~75% of all windows.
    # --- stroke front end (raw sequence -> one vector) ---
    d_conv: int = 64               # width of the temporal conv stack
    kernel: int = 5
    dilations: tuple = (1, 2, 4)   # receptive field ~ 1 + 2*(k-1)*sum(dil) = 29 samples
    d_stroke: int = 128            # per-stroke vector handed to the set encoder
    # --- window encoder ---
    d_hidden: int = 256
    d_embed: int = 64
    n_layers: int = 2
    dropout: float = 0.15
    pool: str = "attn+meanstd"     # "mean" | "meanstd" | "attn+meanstd"
    # --- loss ---
    loss: str = "supcon"           # "supcon" | "batch_hard" | "contrastive"
    temperature: float = 0.1       # supcon only
    margin: float = 0.0            # batch_hard only
    # Both triplet forms collapsed this dataset. They penalise the DIFFERENCE
    # dp - dn, which scales with the embedding radius, so shrinking everything
    # toward a single point reduces the loss monotonically: margin=0.25 parked at
    # loss==0.25, margin=0 parked at softplus(0)=ln2=0.693. supcon is a softmax
    # over cosine similarities and has the opposite behaviour -- a collapsed
    # embedding makes every logit equal, which is its WORST case, not its best.
    # --- optimisation ---
    P_users: int = 8               # users per batch
    K_windows: int = 6             # windows per user per batch  (batch = P*K)
    lr: float = 5e-4               # 2e-3 collapsed the embedding during the warm-up ramp
    weight_decay: float = 1e-2
    epochs: int = 60
    steps_per_epoch: int = 60
    patience: int = 12
    eval_max_windows: int = 3000   # per-epoch validation subsample; full sets in evaluate()
    embed_bs: int = 64
    seed: int = 0


def assign_folds(strokes: pd.DataFrame, val_frac=0.15, test_frac=0.20,
                 seed=0, holdout_users=()) -> pd.DataFrame:
    """Tag every SESSION with train/val/test (or 'unseen' for held-out users).

    Done at session level and before the scaler is fitted, so the fold a window
    belongs to is fixed by its session and nothing leaks across folds.
    """
    rng = np.random.default_rng(seed)
    out = strokes.copy()
    fold = pd.Series("train", index=out.index, dtype=object)
    for u, g in out.groupby("user", sort=True):
        if u in holdout_users:
            fold[g.index] = "unseen"; continue
        us = np.array(sorted(g.session.unique())); rng.shuffle(us)
        n_te = max(1, int(round(len(us) * test_frac)))
        n_va = max(1, int(round(len(us) * val_frac))) if len(us) - n_te >= 2 else 0
        tag = {s: "test" for s in us[:n_te]}
        tag |= {s: "val" for s in us[n_te:n_te + n_va]}
        tag |= {s: "train" for s in us[n_te + n_va:]}
        fold[g.index] = g.session.map(tag).values
    out["fold"] = fold
    print(out.groupby("fold").agg(strokes=("user", "size"), sessions=("session", "nunique"),
                                  users=("user", "nunique")).to_string())
    return out


def build_window_index(df: pd.DataFrame, mcfg: ModelConfig):
    """Sliding window over each session -> (M, n_strokes) row indices into the bank.

    Indices, not data. With stride 8 out of 25 every stroke lands in ~3 windows;
    materialising each window separately would triple the raw points in memory
    for nothing.
    """
    df = df.reset_index(drop=True)
    rows, y, sess, folds, spans, times = [], [], [], [], [], []
    for (u, s), g in df.groupby(["user", "session"], sort=False):
        g = g.sort_values("t_start")
        pos = g.index.to_numpy()
        t0, t1 = g.t_start.to_numpy(), g.t_end.to_numpy()
        for end in range(mcfg.n_strokes, len(g) + 1, mcfg.stride):
            st = end - mcfg.n_strokes
            span = t1[end - 1] - t0[st]
            if span > mcfg.max_span_s:
                continue
            rows.append(pos[st:end]); y.append(u); sess.append(s); spans.append(span)
            times.append((t0[st], t1[end - 1]))
            folds.append(g["fold"].iloc[0] if "fold" in g.columns else "train")
    if not rows:
        raise RuntimeError("no windows -- lower n_strokes or raise max_span_s")
    W = np.stack(rows).astype(np.int64)
    y, sess, folds = np.array(y), np.array(sess), np.array(folds)
    wmeta = np.asarray(times, dtype=np.float64)          # (M, 2): window start / end
    print(f"[windows] {len(W):,} windows x {W.shape[1]} strokes | {len(np.unique(y))} users "
          f"| median span {np.median(spans):.1f}s")
    cnt = pd.Series(y).value_counts()
    print(f"[windows] per-user: min {cnt.min()}, median {int(cnt.median())}, max {cnt.max()}")
    return W, y, sess, folds, wmeta


def fold_index(folds, sess, y):
    out = {f: np.where(folds == f)[0] for f in ("train", "val", "test", "unseen")
           if (folds == f).any()}
    print("[split] " + " | ".join(
        f"{k}: {len(v):,} win / {len(set(sess[v]))} sess / {len(set(y[v]))} users"
        for k, v in out.items()))
    for a in out:
        for b in out:
            if a < b:
                assert not (set(sess[out[a]]) & set(sess[out[b]])), f"session leak {a}/{b}"
    return out


class WindowBank:
    """Stroke bank + window index -> padded (B, N, L, C) batches with a mask.

    Scaling happens here, at batch time, rather than in the stored array: the
    bank holds raw pixels and seconds (exact in float16 -- coordinates are
    integers), so re-fitting the scaler never means rebuilding the bank.
    """

    kind = "raw"

    def __init__(self, S, lens, W, y, sess, folds, scaler=None, device=None):
        self.S, self.lens, self.W = S, lens, W
        self.y, self.sess, self.folds = y, sess, folds
        self.scaler = self.preproc = scaler
        self.device = device or DEVICE
        self._cache = None

    def make_encoder(self, mcfg):
        return RawWindowEncoder(self.n_channels, mcfg)

    def __len__(self):
        return len(self.W)

    @property
    def n_channels(self):
        return self.S.shape[2]

    @property
    def max_len(self):
        return self.S.shape[1]

    def _sc(self):
        if self._cache is None and self.scaler is not None:
            d = self.device
            self._cache = (
                torch.as_tensor(self.scaler["center"], dtype=torch.float32, device=d),
                torch.as_tensor(1.0 / np.asarray(self.scaler["scale"], np.float32),
                                dtype=torch.float32, device=d),
                torch.as_tensor(np.asarray(self.scaler["log"]), dtype=torch.bool, device=d),
            )
        return self._cache

    def batch(self, rows):
        idx = self.W[rows]                                          # (b, N)
        x = torch.from_numpy(np.ascontiguousarray(self.S[idx], dtype=np.float32)).to(self.device)
        n = torch.from_numpy(self.lens[idx].astype(np.int64)).to(self.device)
        ar = torch.arange(self.max_len, device=self.device)
        m = ar[None, None, :] < n[..., None]                        # (b, N, L)
        sc = self._sc()
        if sc is not None:
            center, inv, logm = sc
            if bool(logm.any()):
                x = torch.where(logm, torch.log1p(x.clamp_min(0.0)), x)
            x = (x - center) * inv
        return x * m.unsqueeze(-1), m


class FeatureBank:
    """The hand-crafted stroke features, windowed over the SAME index as the raw
    bank -- identical windows, identical folds, so the only difference between the
    two models is what a stroke is represented by.

    This is the baseline the raw-sequence encoder has to beat. Without it, a
    mediocre number from the raw model is uninterpretable: it could mean raw
    points are the wrong representation, or it could mean the dataset is hard.
    """

    kind = "features"

    def __init__(self, V, W, y, sess, folds, pre, device=None):
        self.V = np.ascontiguousarray(V, dtype=np.float32)
        self.W = W
        self.y, self.sess, self.folds = y, sess, folds
        self.preproc = pre
        self.device = device or DEVICE

    def __len__(self):
        return len(self.W)

    @property
    def n_channels(self):
        return self.V.shape[1]

    @property
    def max_len(self):
        return None                      # one vector per stroke: no time axis

    def make_encoder(self, mcfg):
        return FeatureWindowEncoder(self.n_channels, mcfg)

    def batch(self, rows):
        x = torch.from_numpy(self.V[self.W[rows]]).to(self.device)    # (b, N, D)
        return x, None                   # no mask: nothing is padded


---
## 8.1 Encoder and loss

`batch_hard_triplet` picks, for every anchor in the batch, the *hardest* positive
(same user, furthest away) and the *hardest* negative (different user, closest),
and pushes them apart. Setting `margin=0` switches to the soft-margin form
`log1p(exp(dp - dn))` — no threshold to tune, and it keeps producing gradient after
the easy triplets are exhausted. Worth trying if training plateaus early.

The distance is Euclidean on L2-normalised vectors, which is a monotone function of
cosine, so the metric used in training is the one used at scoring time.

The stroke-level front end sits underneath all of this: `StrokeSeqEncoder`
reduces one variable-length raw stroke to a fixed vector, and only then does the
set encoder above pool 25 of them. The loss never sees the difference.


In [ ]:
# Masked-out logits / maxima. A hard-coded -1e9 raises outright in float16
# (max ~65504), so anything under torch.autocast on a T4 would die here; taking
# the sentinel from the tensor's own dtype keeps fp16 and fp32 paths identical.
def _neg(t):
    return torch.finfo(t.dtype).min


def _pos(t):
    return torch.finfo(t.dtype).max


class AttnPool(nn.Module):
    """Learned-query attention pooling over one axis, mask-aware."""
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))

    def forward(self, h, mask=None):            # h: (B, N, d), mask: (B, N) bool
        s = self.score(h).squeeze(-1)
        if mask is not None:
            s = s.masked_fill(~mask, _neg(s))
        w = torch.softmax(s, dim=1)
        return (h * w.unsqueeze(-1)).sum(1), w


class ResConvBlock(nn.Module):
    """Dilated residual conv over the time axis, re-masked at both ends."""
    def __init__(self, d, k, dil, p):
        super().__init__()
        self.conv = nn.Conv1d(d, d, k, padding=dil * (k - 1) // 2, dilation=dil)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(p)

    def forward(self, h, mf):                   # h: (B, d, L), mf: (B, 1, L) float
        z = self.conv(h * mf)                   # zero the padding before it is convolved in
        z = self.norm(z.transpose(1, 2)).transpose(1, 2)
        return (h + self.drop(F.gelu(z))) * mf


class StrokeSeqEncoder(nn.Module):
    """One variable-length stroke of raw points -> one fixed-size vector.

    Two things this has to survive, both properties of the capture and not of the
    user:

    * **Irregular sampling.** The gap between samples is not constant, so an
      ordinary CNN over the position sequence would be reading a distorted clock.
      Rather than resampling onto a uniform grid -- which invents points that
      were never observed and erases the timing jitter that is itself
      identifying -- `dt` is handed in as an input channel. The network is free
      to learn velocity, or anything else, as a function of (dx, dy, dt).
    * **Variable length.** Strokes run from `min_points` to `max_len` samples.
      Padding is masked out of every reduction below, so a 7-point stroke and a
      120-point stroke are both summarised without the padding contributing.

    Pooling is mean+std+max+attention over time: order *within* a stroke is real
    signal (unlike order within a window), but the pooled summary still has to be
    length-invariant.
    """
    def __init__(self, n_ch, mcfg: ModelConfig):
        super().__init__()
        d = mcfg.d_conv
        self.inp = nn.Linear(n_ch, d)
        self.blocks = nn.ModuleList(
            [ResConvBlock(d, mcfg.kernel, dil, mcfg.dropout) for dil in mcfg.dilations])
        self.attn = AttnPool(d)
        self.out = nn.Sequential(
            nn.Linear(4 * d, mcfg.d_stroke), nn.LayerNorm(mcfg.d_stroke), nn.GELU())

    def forward(self, x, m):                    # x: (B, L, C), m: (B, L) bool
        mf = m.unsqueeze(1).to(x.dtype)         # (B, 1, L)
        h = (self.inp(x).transpose(1, 2)) * mf  # (B, d, L)
        for blk in self.blocks:
            h = blk(h, mf)
        h = h.transpose(1, 2)                   # (B, L, d)

        mb = m.unsqueeze(-1)
        n = m.sum(1, keepdim=True).clamp_min(1).to(x.dtype)
        mean = (h * mb).sum(1) / n
        var = (((h - mean.unsqueeze(1)) ** 2) * mb).sum(1) / n
        std = var.clamp_min(1e-8).sqrt()
        mx = h.masked_fill(~mb, _neg(h)).max(1).values
        att, _ = self.attn(h, m)
        return self.out(torch.cat([att, mean, std, mx], dim=-1))


class StrokeSetEncoder(nn.Module):
    """Window of N per-stroke vectors -> one L2-normalised embedding.

    Per-stroke MLP, then pooling over the stroke axis. Pooling is permutation
    invariant by design: within a 25-stroke window the ordering is close to
    arbitrary, and an order-sensitive encoder (GRU/transformer) mostly learns
    session-specific sequencing, which is exactly the thing that does not
    transfer to a new session.
    """
    def __init__(self, d_in, mcfg: ModelConfig):
        super().__init__()
        d, p = mcfg.d_hidden, mcfg.dropout
        layers, prev = [], d_in
        for _ in range(mcfg.n_layers):
            layers += [nn.Linear(prev, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p)]
            prev = d
        self.stroke = nn.Sequential(*layers)
        self.pool_mode = mcfg.pool
        self.attn = AttnPool(d) if "attn" in mcfg.pool else None
        mult = {"mean": 1, "meanstd": 2, "attn+meanstd": 3}[mcfg.pool]
        self.head = nn.Sequential(
            nn.Linear(d * mult, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p),
            nn.Linear(d, mcfg.d_embed),
        )

    def forward(self, x, return_attn=False):     # x: (B, N, d_in)
        h = self.stroke(x)
        parts = [h.mean(1)]
        if "std" in self.pool_mode:
            parts.append(h.std(1, unbiased=False))
        w = None
        if self.attn is not None:
            a, w = self.attn(h)
            parts.insert(0, a)
        z = F.normalize(self.head(torch.cat(parts, dim=-1)), dim=-1)
        return (z, w) if return_attn else z


class RawWindowEncoder(nn.Module):
    """(B, N, L, C) raw points -> (B, d_embed) L2-normalised window embedding.

    Two levels, and the split is deliberate: *inside* a stroke time order matters
    and the conv stack reads it; *across* strokes in a window it does not, and
    the set encoder throws it away.
    """
    def __init__(self, n_ch, mcfg: ModelConfig):
        super().__init__()
        self.n_ch = n_ch
        self.seq = StrokeSeqEncoder(n_ch, mcfg)
        self.set = StrokeSetEncoder(mcfg.d_stroke, mcfg)

    def forward(self, x, m, return_attn=False):
        B, N, L, C = x.shape
        h = self.seq(x.reshape(B * N, L, C), m.reshape(B * N, L)).view(B, N, -1)
        return self.set(h, return_attn=return_attn)


class FeatureWindowEncoder(nn.Module):
    """Baseline encoder: the hand-crafted per-stroke feature vectors straight into
    the same set encoder, same pooling, same loss, same windows.

    It takes (x, m) like RawWindowEncoder and ignores the mask, so both models
    are interchangeable everywhere downstream -- train_siamese, embed, evaluate
    and save_model never branch on which one they were handed.
    """
    def __init__(self, d_in, mcfg: ModelConfig):
        super().__init__()
        self.n_ch = d_in
        self.set = StrokeSetEncoder(d_in, mcfg)

    def forward(self, x, m=None, return_attn=False):
        return self.set(x, return_attn=return_attn)


def pdist(z):
    """Euclidean distance on L2-normalised embeddings == sqrt(2-2cos)."""
    return torch.cdist(z, z, p=2).clamp_min(0)


def batch_hard_triplet(z, labels, margin=0.25):
    """For each anchor: hardest positive, hardest negative, inside the batch.

    margin <= 0 switches to the soft-margin form log1p(exp(dp - dn)), which has
    no threshold to tune and does not go dead once easy triplets are exhausted.
    """
    D = pdist(z)
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    pos = same & ~eye
    neg = ~same
    valid = pos.any(1) & neg.any(1)
    if not valid.any():
        return z.sum() * 0.0, {}
    dp = (D.masked_fill(~pos, _neg(D))).max(1).values[valid]        # hardest positive
    dn = (D.masked_fill(~neg, _pos(D))).min(1).values[valid]       # hardest negative
    loss = F.relu(dp - dn + margin).mean() if margin > 0 else F.softplus(dp - dn).mean()
    return loss, {"dp": dp.mean().item(), "dn": dn.mean().item(),
                  "viol": (dp + margin > dn).float().mean().item()}


def supcon(z, labels, temperature=0.1):
    """Supervised contrastive loss: softmax over cosine similarities, every
    same-user window in the batch counted as a positive.

    Why this and not triplet. Triplet minimises dp - dn, which is a difference of
    distances and therefore shrinks when the whole embedding shrinks -- collapse
    is a *descent direction*. Here the logits are z_i . z_j / T, and a collapsed
    embedding makes every logit identical, so the softmax becomes uniform and the
    loss hits its maximum log(B-1). The degenerate solution is the worst one
    reachable, not the easiest.

    The temperature sets how hard the ranking is pushed; 0.05-0.2 is the usual
    range, lower being sharper.
    """
    sim = (z @ z.t()) / temperature
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, _neg(sim))
    logprob = sim - torch.logsumexp(sim, dim=1, keepdim=True)

    pos = (labels[:, None] == labels[None, :]) & ~eye
    cnt = pos.sum(1)
    valid = cnt > 0
    if not valid.any():
        return z.sum() * 0.0, {}
    loss = -(logprob * pos).sum(1)[valid] / cnt[valid]

    with torch.no_grad():
        cos = z @ z.t()
        neg = ~pos & ~eye
        sp = cos[pos].mean().item() if pos.any() else float("nan")
        sn = cos[neg].mean().item() if neg.any() else float("nan")
    return loss.mean(), {"cos_pos": sp, "cos_neg": sn, "gap": sp - sn}


def contrastive_pairs(z, labels, margin=1.0):
    """Classic siamese pair loss, kept for comparison against batch_hard."""
    D = pdist(z)
    same = (labels[:, None] == labels[None, :]).float()
    iu = torch.triu_indices(len(z), len(z), offset=1, device=z.device)
    d, s = D[iu[0], iu[1]], same[iu[0], iu[1]]
    loss = (s * d.pow(2) + (1 - s) * F.relu(margin - d).pow(2)).mean()
    return loss, {"d_pos": d[s > 0].mean().item(), "d_neg": d[s == 0].mean().item()}


class PKSampler:
    """P users x K windows per batch -- without it, random batches rarely contain
    a positive pair and the mining above has nothing to mine."""
    def __init__(self, y, sess, P, K, steps, seed=0):
        self.rng = np.random.default_rng(seed)
        self.P, self.K, self.steps = P, K, steps
        self.by_user = {u: np.where(y == u)[0] for u in np.unique(y)}
        self.sess = sess
        self.users = [u for u, v in self.by_user.items() if len(v) >= 2]

    def __iter__(self):
        for _ in range(self.steps):
            P = min(self.P, len(self.users))
            picked = self.rng.choice(self.users, size=P, replace=False)
            batch = []
            for u in picked:
                pool = self.by_user[u]
                # spread the K windows over distinct sessions where possible, so the
                # "hardest positive" is a cross-session pair rather than a neighbour
                sess_u = self.sess[pool]
                order = self.rng.permutation(len(pool))
                seen, first, rest = set(), [], []
                for i in order:
                    (first if sess_u[i] not in seen else rest).append(pool[i])
                    seen.add(sess_u[i])
                take = (first + rest)[: self.K]
                if len(take) < self.K:
                    take = list(self.rng.choice(pool, size=self.K, replace=True))
                batch += take
            yield np.array(batch)

    def __len__(self):
        return self.steps

---
## 8.2 Metrics

Three numbers, measuring three different questions:

- **rank-1** (`template_metrics`) — closed-set identification: of the enrolled users,
  who is this? Enrolment comes from train sessions, probes from held-out sessions,
  so it is cross-session by construction.
- **EER** — verification: at what error rate do false accepts equal false rejects?
  This is the number the Balabit literature reports and the one that matters for an
  authentication system.
- **session-level** — average all windows of a session into one embedding first.
  A real system sees a whole session, not 25 strokes, and this is always far better
  than the per-window figure. Quote both; quoting only the session number is how
  papers get to "99.9%".

`pair_metrics` restricts genuine pairs to **different sessions**. Two windows from
the same session are minutes apart, share a posture, a mouse and a mood, and will
match on things that have nothing to do with identity. Including them inflates every
figure and is the single most common way these results get overstated.

In [ ]:
from sklearn.metrics import roc_curve, auc as _auc


def eer_from_scores(y_true, score):
    """score = similarity (higher == more likely genuine)."""
    fpr, tpr, thr = roc_curve(y_true, score)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2), float(_auc(fpr, tpr)), float(thr[i])


@torch.no_grad()
def embed(enc, bank: "WindowBank", idx=None, bs=64):
    """Embed windows straight out of the bank -- batches are built on demand, so
    the raw points are never materialised per window."""
    enc.eval()
    rows = np.arange(len(bank)) if idx is None else np.asarray(idx)
    out = []
    for i in range(0, len(rows), bs):
        x, m = bank.batch(rows[i:i + bs])
        out.append(enc(x, m).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 1), np.float32)


def templates(z, y):
    """One L2-normalised centroid per user (the 'enrolment' step)."""
    us = np.array(sorted(set(y)))
    T = np.stack([z[y == u].mean(0) for u in us])
    return T / np.linalg.norm(T, axis=1, keepdims=True), us


def template_metrics(z_enroll, y_enroll, z_probe, y_probe, tag=""):
    """Probe each window against every user's enrolment template.

    This is the protocol that matters: enrolment and probe come from different
    sessions by construction, so it cannot be gamed by within-session similarity.
    """
    T, us = templates(z_enroll, y_enroll)
    S = z_probe @ T.T                                   # cosine similarity
    keep = np.isin(y_probe, us)
    S, yp = S[keep], y_probe[keep]
    gt = np.array([list(us).index(u) for u in yp])
    rank1 = float((S.argmax(1) == gt).mean())
    lab = np.zeros_like(S, dtype=int); lab[np.arange(len(gt)), gt] = 1
    eer, auc_, thr = eer_from_scores(lab.ravel(), S.ravel())
    m = {"rank1": rank1, "eer": eer, "auc": auc_, "thr": thr, "n_probe": int(len(yp))}
    if tag:
        print(f"[{tag}] rank-1 {rank1:6.1%}   EER {eer:6.2%}   AUC {auc_:.4f}   "
              f"(n={len(yp)}, {len(us)} users)")
    return m


def pair_metrics(z, y, sess, cross_session_only=True, max_pairs=200_000, seed=0, tag=""):
    """Window-vs-window verification.

    Genuine pairs are restricted to DIFFERENT sessions. Same-session pairs are
    minutes apart at most and inflate every number you would report.
    """
    rng = np.random.default_rng(seed)
    n = len(z)
    iu = np.triu_indices(n, 1)
    if len(iu[0]) > max_pairs:
        sel = rng.choice(len(iu[0]), max_pairs, replace=False)
        iu = (iu[0][sel], iu[1][sel])
    i, j = iu
    same_u, same_s = y[i] == y[j], sess[i] == sess[j]
    keep = ~same_s if cross_session_only else np.ones(len(i), bool)
    s = (z[i] * z[j]).sum(1)[keep]
    lab = same_u[keep].astype(int)
    if lab.sum() == 0 or (1 - lab).sum() == 0:
        return {"eer": np.nan, "auc": np.nan, "n_pairs": int(keep.sum())}
    eer, auc_, thr = eer_from_scores(lab, s)
    m = {"eer": eer, "auc": auc_, "thr": thr, "n_pairs": int(keep.sum()),
         "n_genuine": int(lab.sum())}
    if tag:
        print(f"[{tag}] EER {eer:6.2%}   AUC {auc_:.4f}   "
              f"({lab.sum():,} genuine / {(1-lab).sum():,} impostor pairs)")
    return m


def session_level(z, y, sess, z_enroll, y_enroll, tag=""):
    """Average a session's windows into one embedding before deciding.

    A real system gets a whole session, not one 25-stroke window; this is the
    number to quote for 'can we tell who is at the keyboard', and it is always
    much better than the per-window number.
    """
    keys = sorted(set(zip(y, sess)))
    Z = np.stack([z[(y == u) & (sess == s)].mean(0) for u, s in keys])
    Z /= np.linalg.norm(Z, axis=1, keepdims=True)
    yy = np.array([u for u, _ in keys])
    return template_metrics(z_enroll, y_enroll, Z, yy, tag=tag)


import matplotlib.pyplot as plt


def plot_embeddings(z, y, title="test embeddings (PCA)"):
    zc = z - z.mean(0)
    U, S, Vt = np.linalg.svd(zc, full_matrices=False)
    P = zc @ Vt[:2].T
    fig, ax = plt.subplots(figsize=(6, 5))
    for u in sorted(set(y)):
        m = y == u
        ax.scatter(P[m, 0], P[m, 1], s=14, alpha=.75, label=u)
    ax.set_title(f"{title}  ({S[:2].sum()/S.sum():.0%} of variance)")
    ax.legend(fontsize=7, ncol=2, markerscale=1.5)
    plt.tight_layout(); plt.show()

---
## 8.3 Train

Early stopping on validation EER, not on loss.

**Read `spread` first, before any metric.** It is the mean distance of a validation
embedding from the centroid of all of them. If it decays toward zero the model has
collapsed to a constant and every number below it is noise, whatever the loss says.
`train_siamese` prints a `COLLAPSED` warning under 0.05.

### The collapse, and what actually fixed it

The first full run collapsed on a batch-hard triplet loss with `margin=0.25`:
`dp` fell 0.227 → 0.006, `dn` 0.156 → 0.004, `viol` stuck at 1.0, and the loss
parked at 0.2520 —

    relu(dp - dn + margin) = 0.00635 - 0.00431 + 0.25 = 0.25204

the reported loss to five digits. Switching to the soft margin (`margin=0`) did
**not** help: it parked at 0.6940 instead, which is `softplus(0) = ln 2 = 0.6931`,
the same degeneracy at a different constant, with spread down at 0.0012.

The reason is structural, and it is worth internalising before reaching for
another triplet variant. Both forms penalise the *difference* `dp - dn`. That
difference scales with the radius of the embedding cloud, so shrinking everything
toward a single point reduces the loss monotonically — **collapse is a descent
direction**, not a local minimum the optimiser stumbles into. No learning rate or
margin fixes a loss whose easiest minimiser is the trivial one.

`supcon` (the default) is a softmax over cosine similarities at temperature `T`.
A collapsed embedding makes every logit identical, the softmax uniform, and the
loss equal to its *maximum* `log(B-1)`. The degenerate solution is the worst
point reachable rather than the cheapest. Measured on six users, 25 epochs:

| loss | val EER | val rank-1 | final spread |
|---|---|---|---|
| triplet, `margin=0.25` | 30.99% | 38.1% | collapsed |
| triplet, `margin=0` | 30.73% | 43.6% | 0.0012 |
| **supcon, `T=0.1`** | **25.28%** | **52.5%** | **0.375** |

Watch `cos+`, `cos-` and `gap` in the log. The gap is the signal — it should grow.
Both absolute values drifting toward 1.0 together is the collapse signature again.

### The other changes

| was | now | why |
|---|---|---|
| `lr=2e-3` | `lr=5e-4` | the original collapse completed by epoch 5, during the OneCycle warm-up ramp |
| `n_strokes=25, stride=8, max_span_s=90` | `12 / 4 / 240` | at Balabit's ~9 Hz a 25-stroke window routinely spans minutes, so the span filter was silently discarding ~75% of all windows — 4,878 survived out of ~21,000 |

Both representations train here over identical windows and folds: raw points and
the hand-crafted-feature baseline. One run, two numbers. Without the baseline a
mediocre raw number cannot be attributed to the representation rather than to the
dataset — and on this data the baseline is currently winning.

In [ ]:
def _cap(rows, cap, seed=0):
    """Fixed random subsample, so the per-epoch validation signal is comparable
    across epochs. The full sets are used in evaluate()."""
    rows = np.asarray(rows)
    if cap and len(rows) > cap:
        return np.sort(np.random.default_rng(seed).choice(rows, cap, replace=False))
    return rows


def train_siamese(bank, idx, mcfg: ModelConfig, verbose_every=5):
    """Works on either bank: the encoder comes from bank.make_encoder()."""
    torch.manual_seed(mcfg.seed); np.random.seed(mcfg.seed)
    y, sess = bank.y, bank.sess
    tr, va = idx["train"], idx.get("val", idx["train"])

    enc = bank.make_encoder(mcfg).to(DEVICE)
    n_par = sum(p.numel() for p in enc.parameters())
    shape = f"{bank.n_channels}ch x {bank.max_len}pts" if bank.max_len else \
            f"{bank.n_channels} features"
    print(f"[model:{bank.kind}] {shape} x {mcfg.n_strokes} strokes -> "
          f"{mcfg.d_embed}d window  ({n_par/1e6:.2f}M params)")

    opt = torch.optim.AdamW(enc.parameters(), lr=mcfg.lr, weight_decay=mcfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=mcfg.lr, total_steps=mcfg.epochs * mcfg.steps_per_epoch, pct_start=0.2)

    ytr, str_ = y[tr], sess[tr]
    sampler = PKSampler(ytr, str_, mcfg.P_users, mcfg.K_windows, mcfg.steps_per_epoch, mcfg.seed)
    lut = {u: i for i, u in enumerate(sorted(set(ytr)))}

    # subsampled enrol/probe sets for the per-epoch metric -- embedding every
    # window through the conv stack each epoch would cost more than training does
    tr_ev = _cap(tr, mcfg.eval_max_windows, mcfg.seed)
    va_ev = _cap(va, mcfg.eval_max_windows, mcfg.seed + 1)

    best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(1, mcfg.epochs + 1):
        enc.train(); tot, stats, t0 = 0.0, {}, time.time()
        for b in sampler:
            rows = tr[b]                                   # sampler is local to tr
            xb, mb = bank.batch(rows)
            lb = torch.as_tensor([lut[u] for u in ytr[b]], device=DEVICE)
            z = enc(xb, mb)
            if mcfg.loss == "supcon":
                loss, st = supcon(z, lb, mcfg.temperature)
            elif mcfg.loss == "batch_hard":
                loss, st = batch_hard_triplet(z, lb, mcfg.margin)
            else:
                loss, st = contrastive_pairs(z, lb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 5.0)
            opt.step(); sched.step()
            tot += loss.item(); stats = st

        z_tr = embed(enc, bank, tr_ev, mcfg.embed_bs)
        z_va = embed(enc, bank, va_ev, mcfg.embed_bs)
        m = template_metrics(z_tr, y[tr_ev], z_va, y[va_ev])
        # spread = how far apart the embeddings actually are. If this decays toward
        # zero the model is collapsing to a constant, whatever the loss says.
        spread = float(np.linalg.norm(z_va - z_va.mean(0), axis=1).mean())
        hist.append({"epoch": ep, "loss": tot / mcfg.steps_per_epoch,
                     "val_eer": m["eer"], "val_rank1": m["rank1"],
                     "spread": spread, "sec": time.time() - t0, **stats})
        if verbose_every and (ep % verbose_every == 0 or ep == 1):
            print(f"  ep {ep:3d}  loss {hist[-1]['loss']:.4f}  "
                  f"val EER {m['eer']:6.2%}  val rank-1 {m['rank1']:6.1%}  "
                  f"spread {spread:.3f}"
                  + (f"  dp {stats['dp']:.3f} dn {stats['dn']:.3f}" if "dp" in stats else "")
                  + (f"  cos+ {stats['cos_pos']:.3f} cos- {stats['cos_neg']:.3f}"
                     f" gap {stats['gap']:+.3f}" if "gap" in stats else "")
                  + f"  ({hist[-1]['sec']:.0f}s)")

        if m["eer"] < best - 1e-4:
            best, bad = m["eer"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()}
        else:
            bad += 1
            if bad >= mcfg.patience:
                print(f"  early stop at epoch {ep} (best val EER {best:.2%})")
                break

    if best_state is not None:
        enc.load_state_dict(best_state)
    h = pd.DataFrame(hist)
    print(f"[train:{bank.kind}] best val EER {best:.2%}   "
          f"final spread {h.spread.iloc[-1]:.4f}"
          + ("   <-- COLLAPSED: embeddings are ~one point, treat the metrics as noise"
             if h.spread.iloc[-1] < 0.05 else ""))
    return enc, h


def evaluate(enc, bank, idx, mcfg: ModelConfig):
    y, sess = bank.y, bank.sess
    tr = idx["train"]
    z_tr = embed(enc, bank, tr, mcfg.embed_bs)
    out = {}
    for fold in ("val", "test"):
        if fold not in idx:
            continue
        f = idx[fold]
        z = embed(enc, bank, f, mcfg.embed_bs)
        out[f"{fold}_window"]  = template_metrics(z_tr, y[tr], z, y[f], tag=f"{fold} window")
        out[f"{fold}_session"] = session_level(z, y[f], sess[f], z_tr, y[tr], tag=f"{fold} session")
        out[f"{fold}_pairs"]   = pair_metrics(z, y[f], sess[f], tag=f"{fold} pairs  ")
    if "unseen" in idx:
        u = idx["unseen"]
        zu = embed(enc, bank, u, mcfg.embed_bs)
        print("\n-- users never seen in training (open-set: the honest number) --")
        out["unseen_pairs"] = pair_metrics(zu, y[u], sess[u], tag="unseen pairs")
    return out


# ============================================================================
mcfg = ModelConfig()          # see the dataclass for what changed and why
# rate_norm_hz=9.0 measured FREE on the full set: raw test EER 22.94% -> 22.50%,
# rank-1 45.5% -> 52.1%. It only thins 7.6% of strokes (most are already ~9 Hz) and
# it stops `dt` being a machine fingerprint, so deployment on a different capture
# rate is no longer an untested assumption. Set to None to train on native rates.
scfg = SeqConfig(max_len=64, rate_norm_hz=9.0)

# A/B on the machine-fingerprint hypothesis: absolute screen position is as much a
# property of the monitor as of the person, and in Balabit the capture rate tracks
# the user. Swap these two lines to find out how much of the score is the machine.
scfg.channels = ("dx", "dy", "dt", "x_rel", "y_rel", "x_abs", "y_abs", "is_drag")
# scfg.channels = ("dx", "dy", "dt", "x_rel", "y_rel", "is_drag")

# Two users held out entirely -> the open-set block in 8.4. Set to () to train on all.
HOLDOUT_USERS = tuple(sorted(strokes.user.unique())[-2:])
print("holding out:", HOLDOUT_USERS)

# folds first (session level), then the bank, then the scalers on train points only
st_f = assign_folds(strokes, val_frac=0.15, test_frac=0.20, seed=0,
                    holdout_users=HOLDOUT_USERS)
S, lens, st_f = build_stroke_bank(st_f, SEQS, scfg)
W, y, sess, folds, wmeta = build_window_index(st_f, mcfg)
idx = fold_index(folds, sess, y)

# --- the two representations, over identical windows ------------------------
seq_scaler = fit_seq_scaler(S, lens, np.unique(W[idx["train"]]), scfg, seed=mcfg.seed)
bank_raw = WindowBank(S, lens, W, y, sess, folds, scaler=seq_scaler)

pre_m = fit_preprocessor(st_f[st_f.fold == "train"], PREP)    # train fold only
V = apply_preprocessor(st_f, pre_m).to_numpy(dtype=np.float32)
bank_feat = FeatureBank(V, W, y, sess, folds, pre_m)

BANKS = {"raw": bank_raw, "features": bank_feat}    # delete an entry to skip it

models, hists = {}, {}
for name, bk in BANKS.items():
    print(f"\n{'='*78}\n== {name}\n{'='*78}")
    models[name], hists[name] = train_siamese(bk, idx, mcfg)

# The feature encoder is the primary model, not the raw one. Four independent
# comparisons now put it ahead: 10 users 17.6% vs 22.5% test EER, and even on the
# 63 Hz sessions where the raw encoder is at its best, 2.8% vs 4.0%. Swap the key
# to "raw" if you want the sequence model in the downstream cells instead.
PRIMARY = "features"
enc, hist, bank = models[PRIMARY], hists[PRIMARY], BANKS[PRIMARY]

---
## 8.4 Evaluate

If `HOLDOUT_USERS` is non-empty, the last block is the honest number: users the
model has never seen, scored purely on whether the embedding puts *any* two windows
from the same stranger closer together than windows from different strangers. That
is what an embedding model is for, and it is normally a few points worse than the
seen-user figures. With only 10 Balabit users, holding 2 out costs real training
signal — worth running once to get the number, then training on all 10 for the
deployed model.

In [ ]:
res = {}
for name, bk in BANKS.items():
    print(f"\n{'='*78}\n== {name}\n{'='*78}")
    res[name] = evaluate(models[name], bk, idx, mcfg)

# head-to-head on the numbers that matter
rows = []
for name in BANKS:
    r = res[name]
    rows.append({
        "model": name,
        "test rank-1": r["test_window"]["rank1"],
        "test win EER": r["test_window"]["eer"],
        "test sess EER": r["test_session"]["eer"],
        "unseen EER": r.get("unseen_pairs", {}).get("eer", np.nan),
        "collapsed?": "YES" if hists[name].spread.iloc[-1] < 0.05 else "no",
    })
cmp = pd.DataFrame(rows).set_index("model")
print("\n" + "="*78)
print("RAW SEQUENCES vs HAND-CRAFTED FEATURES  (identical windows, folds and loss)")
print("="*78)
print(cmp.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"\nchance rank-1 = {1/len(set(y)):.1%};  EER 50% = coin flip, lower is better")

z_test = embed(enc, bank, idx["test"], mcfg.embed_bs)
plot_embeddings(z_test, y[idx["test"]], title="raw-sequence test embeddings (PCA)")

display(hist.tail(10))

---
## 8.5 Decision layer: how much activity does a verdict need?

Everything above scores one 12-stroke window — roughly 15–30 seconds of activity —
against a user centroid with plain cosine. That is not the question the product
asks. Continuous authentication gets to *accumulate*: it can watch for two
minutes before committing, and it runs at a chosen operating point rather than at
EER.

Three changes, none of which require retraining:

**Accumulate.** `decision_sweep` averages k consecutive windows from one session
before scoring. The windows are consecutive and in order, never randomly drawn
across the session — random draws would quietly assume the errors are independent
and make the curve look better than the system is.

**Normalise the scores.** Cosine-to-centroid is biased per user: a template in a
dense part of the sphere attracts high scores from everybody, so one global
threshold is too strict for one user and too loose for another. `s-norm`
standardises each template by the impostor scores it attracts (z-norm) and each
probe across templates (t-norm), and averages the two. The cohort statistics are
fitted on the **train fold** — they are fitted parameters like any other.

**Fuse.** Both encoders are trained on identical windows, and they see different
things: one reads temporal shape, the other 60 summary statistics. Averaging two
normalised score matrices costs nothing and is a reliable gain when the views are
decorrelated.

### How to read the output

The EER-vs-time curve is the fork in the road, and either answer is worth having:

* **Falls steeply toward single digits** → the embedding is good enough, the
  remaining work is productisation, and you should stop tuning the encoder.
* **Plateaus around 15–20%** → errors are session-correlated rather than
  independent, accumulation cannot rescue it, and the representation is the thing
  to fix.

`far@frr5` is the number to quote to anyone building on this: *at a setting where
we wrongly challenge the real user 5% of the time, how often does an impostor get
through?*

In [ ]:
# ---------------------------------------------------------------------------
# Decision layer: accumulate evidence over time, normalise scores, fuse models
# ---------------------------------------------------------------------------


def chunk_windows(y, sess, k):
    """Runs of k consecutive windows from one session, non-overlapping.

    Consecutive and in order, not randomly drawn: a deployed system accumulates
    whatever the user happens to do next, and neighbouring windows are correlated.
    Sampling randomly across a session would quietly assume independence and make
    the curve below look better than the system is.
    """
    out, n, start = [], len(y), 0
    for i in range(1, n + 1):
        if i == n or sess[i] != sess[start] or y[i] != y[start]:
            seg = np.arange(start, i)
            for j in range(0, len(seg) - k + 1, k):
                out.append(seg[j:j + k])
            start = i
    return out


def build_scorer(z_enroll, y_enroll, mode="snorm", cohort=None, y_cohort=None):
    """Templates + the impostor-score statistics that s-norm needs.

    Plain cosine-to-centroid is biased per user: some templates sit in a dense
    part of the sphere and everything looks close to them, so one global
    threshold is simultaneously too strict for one user and too loose for
    another. z-norm rescales each template by the spread of *impostor* scores it
    attracts; t-norm rescales each probe across templates; s-norm averages both.

    The cohort must come from the train fold -- it is fitted statistics like any
    other, and fitting it on the probes would leak.
    """
    T, us = templates(z_enroll, y_enroll)
    sc = {"T": T, "users": us, "mode": mode}
    if mode in ("znorm", "snorm"):
        zc = z_enroll if cohort is None else cohort
        yc = y_enroll if cohort is None else y_cohort
        S = zc @ T.T
        mu = np.zeros(len(us)); sd = np.ones(len(us))
        for j, u in enumerate(us):
            imp = S[yc != u, j]                     # impostors against this template
            if imp.size > 1:
                mu[j], sd[j] = float(imp.mean()), float(imp.std()) + 1e-9
        sc["mu"], sc["sd"] = mu, sd
    return sc


def score(sc, z):
    """(n, d) embeddings -> (n, n_users) normalised similarity."""
    S = z @ sc["T"].T
    parts = []
    if sc["mode"] in ("znorm", "snorm"):
        parts.append((S - sc["mu"]) / sc["sd"])
    if sc["mode"] in ("tnorm", "snorm"):
        parts.append((S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9))
    return np.mean(parts, axis=0) if parts else S


def operating_points(lab, s, frrs=(0.01, 0.05)):
    """EER plus FAR at fixed FRR -- a deployed system runs at an operating point,
    and 'how often do we lock out the real user' is the constraint the product has.
    """
    fpr, tpr, thr = roc_curve(lab, s)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    out = {"eer": float((fpr[i] + fnr[i]) / 2), "auc": float(_auc(fpr, tpr)),
           "thr_eer": float(thr[i])}
    for t in frrs:
        k = np.where(fnr <= t)[0]
        out[f"far@frr{int(t*100)}"] = float(fpr[k[0]]) if len(k) else np.nan
    return out


def decision_sweep(models, banks, idx, mcfg, wmeta, ks=(1, 2, 4, 8, 16, 32, 64),
                   fold="test", mode="snorm", fuse=True, min_chunks=25):
    """EER as a function of how much activity the decision is allowed to see.

    This is the fork in the road. If EER falls steeply with k, the embedding is
    good enough and the remaining work is productisation. If it plateaus, the
    errors are session-correlated rather than independent, no amount of
    accumulation rescues it, and the representation is the thing to fix.
    """
    any_bank = next(iter(banks.values()))
    y, sess = any_bank.y, any_bank.sess
    tr, te = idx["train"], idx[fold]

    Z, scorers = {}, {}
    for name, bk in banks.items():
        z_tr = embed(models[name], bk, tr, mcfg.embed_bs)
        Z[name] = (z_tr, embed(models[name], bk, te, mcfg.embed_bs))
        scorers[name] = build_scorer(z_tr, y[tr], mode=mode, cohort=z_tr, y_cohort=y[tr])

    us = scorers[next(iter(scorers))]["users"]
    rows = []
    for k in ks:
        chunks = chunk_windows(y[te], sess[te], k)
        if len(chunks) < min_chunks:
            continue
        yc = np.array([y[te][ch[0]] for ch in chunks])
        # real observed seconds: first window's start to last window's end
        t0 = wmeta[te][:, 0]; t1 = wmeta[te][:, 1]
        secs = float(np.median([t1[ch[-1]] - t0[ch[0]] for ch in chunks]))

        S_all = {}
        for name, (_, z_te) in Z.items():
            zz = np.stack([z_te[ch].mean(0) for ch in chunks])
            zz /= np.linalg.norm(zz, axis=1, keepdims=True)
            S_all[name] = score(scorers[name], zz)
        if fuse and len(S_all) > 1:
            S_all["fusion"] = np.mean(list(S_all.values()), axis=0)

        keep = np.isin(yc, us)
        lab = (yc[keep][:, None] == us[None, :]).astype(int).ravel()
        for name, S in S_all.items():
            m = operating_points(lab, S[keep].ravel())
            m.update({"model": name, "k": k, "median_s": secs,
                      "n_trials": int(lab.size), "n_probes": int(keep.sum()),
                      # large k needs long sessions, so the surviving probes are a
                      # biased, shrinking sample -- an EER of 0 off 40 probes is an
                      # empty sample, not a perfect system
                      "reliable": int(keep.sum()) >= 150})
            rows.append(m)
    return pd.DataFrame(rows)


def plot_sweep(sw):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for name, g in sw.groupby("model"):
        ax[0].plot(g.median_s / 60, 100 * g.eer, "o-", label=name)
        ax[1].plot(g.median_s / 60, 100 * g["far@frr5"], "o-", label=name)
    for a, t in zip(ax, ["EER (%)", "FAR (%) at FRR = 5%"]):
        a.set_xscale("log"); a.set_xlabel("observation time (minutes, median)")
        a.set_ylabel(t); a.grid(alpha=.3); a.legend()
    ax[0].axhline(10, ls="--", c="grey", lw=1)
    ax[0].annotate("10% EER", (ax[0].get_xlim()[0], 10.5), fontsize=8, color="grey")
    fig.suptitle("Does accumulating more activity fix it?")
    plt.tight_layout(); plt.show()


# --- run it -----------------------------------------------------------------
SWEEP_K = (1, 2, 4, 8, 16, 32, 64)

print("baseline: plain cosine to centroid, no normalisation")
sw_raw = decision_sweep(models, BANKS, idx, mcfg, wmeta, ks=SWEEP_K, mode="none")
print(sw_raw.pivot_table(index="k", columns="model", values="eer").round(4).to_string())

print("\nwith s-norm:")
sweep = decision_sweep(models, BANKS, idx, mcfg, wmeta, ks=SWEEP_K, mode="snorm")

cols = ["model", "k", "median_s", "n_probes", "reliable", "eer", "auc", "far@frr1", "far@frr5"]
print(sweep[cols].to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

rel = sweep[sweep.reliable]
best = rel.loc[rel.groupby("model").eer.idxmin()]   # ignore the unreliable tail
print("\nbest operating point per model (reliable rows only, n_probes >= 150):")
for _, r in best.iterrows():
    print(f"  {r['model']:9s} EER {r.eer:6.2%} at k={int(r.k):3d} "
          f"({r.median_s/60:5.1f} min)   FAR {r['far@frr5']:6.2%} at FRR 5%")

base = sw_raw.set_index(["model", "k"]).eer
delta = sweep.set_index(["model", "k"]).eer - base          # positive == s-norm WORSE
print(f"\ns-norm effect on EER: {delta.mean():+.4f} absolute, "
      f"{100 * (delta / base).replace([np.inf, -np.inf], np.nan).mean():+.1f}% relative "
      f"({'WORSE' if delta.mean() > 0 else 'better'}; s-norm cohort statistics need many "
      f"more identities than {len(best)} to be stable)")

plot_sweep(sweep)

---
## 8.7 Sampling-rate ablation

Two claims have been made about this dataset and neither has been tested:
that ~9 Hz capture is what holds the raw-sequence encoder back, and that the
rate/user correlation is inflating the scores. Across the full dataset they
cannot be separated — rate is confounded with identity, so "slower capture" and
"different person" are the same variable.

The fast sessions break the confound. Take the ~211 sessions captured at 63 Hz,
train on them natively, then decimate **those same recordings** to 9 Hz and train
again. Same people, same sessions, same everything; only the sampling rate moves.
Whatever gap appears is the cost of the sampling rate, measured rather than
assumed.

Note the direction of possible information: you can decimate but never
interpolate, which is why the fast sessions are the only ones that can answer
this.

`SeqConfig.rate_norm_hz` is also the fix, not just the instrument. Setting it in
the main driver equalises every stroke toward a common rate, so `dt` stops being
a machine fingerprint the encoder can read instead of behaviour.

In [ ]:
# ---------------------------------------------------------------------------
# Sampling-rate ablation: is 9 Hz the ceiling, or is it the identity count?
# ---------------------------------------------------------------------------


def run_variant(st, scfg, mcfg, tag, which=("raw", "features"), seed=0, quiet=True):
    """Fold -> bank -> scaler -> train -> evaluate, for one configuration.

    Everything the main driver does, packaged so two configurations can be run
    back to back on the same strokes with only `scfg` differing.
    """
    stv = assign_folds(st, val_frac=0.15, test_frac=0.20, seed=seed, holdout_users=())
    S, lens, stv = build_stroke_bank(stv, SEQS, scfg)
    W, yv, sv, fv, _wm = build_window_index(stv, mcfg)
    ix = fold_index(fv, sv, yv)

    banks = {}
    if "raw" in which:
        sc = fit_seq_scaler(S, lens, np.unique(W[ix["train"]]), scfg, seed=seed)
        banks["raw"] = WindowBank(S, lens, W, yv, sv, fv, scaler=sc)
    if "features" in which:
        pre = fit_preprocessor(stv[stv.fold == "train"], PREP)
        banks["features"] = FeatureBank(
            apply_preprocessor(stv, pre).to_numpy(dtype=np.float32), W, yv, sv, fv, pre)

    out = []
    for name, bk in banks.items():
        enc_v, h = train_siamese(bk, ix, mcfg, verbose_every=0 if quiet else 10)
        r = evaluate(enc_v, bk, ix, mcfg)
        out.append({"variant": tag, "model": name,
                    "test_rank1": r["test_window"]["rank1"],
                    "test_eer": r["test_window"]["eer"],
                    "test_sess_eer": r["test_session"]["eer"],
                    "val_eer_best": float(h.val_eer.min()),
                    "spread": float(h.spread.iloc[-1]),
                    "n_windows": len(bk), "users": int(len(set(yv)))})
    return pd.DataFrame(out)


def rate_ablation(strokes, qc, mcfg, fast_hz=33.0, epochs=None):
    """Same sessions, same users, two sampling rates -- the only clean way to ask
    whether rate matters.

    Across the full dataset rate is confounded with user, so a between-session
    comparison cannot separate "this capture is faster" from "this is a different
    person". Restricting to the fast sessions and decimating *those same
    recordings* makes it a within-subject comparison, where the confound cannot
    apply.
    """
    m = replace(mcfg, epochs=epochs or mcfg.epochs)
    fast = set(map(tuple, qc.loc[qc.get("hz", pd.Series(dtype=float)) > fast_hz,
                                 ["user", "session"]].to_numpy()))
    if not fast:
        raise RuntimeError("no fast sessions found -- check qc['hz']")
    key = list(zip(strokes.user, strokes.session))
    st_fast = strokes[[k in fast for k in key]].reset_index(drop=True)
    print(f"[ablation] {len(fast)} fast sessions, {len(st_fast):,} strokes, "
          f"{st_fast.user.nunique()} users")

    native = replace(scfg, rate_norm_hz=None)
    slowed = replace(scfg, rate_norm_hz=9.0)
    a = run_variant(st_fast, native, m, "native (~63 Hz)")
    b = run_variant(st_fast, slowed, m, "decimated to 9 Hz")
    res = pd.concat([a, b], ignore_index=True)

    print("\n" + "=" * 78)
    print("SAMPLING-RATE ABLATION -- identical sessions, identical users")
    print("=" * 78)
    print(res.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
    piv = res.pivot_table(index="model", columns="variant", values="test_eer")
    if piv.shape[1] == 2:
        piv["cost of 9 Hz"] = piv.iloc[:, 1] - piv.iloc[:, 0]
        print("\ntest EER by rate:")
        print(piv.round(4).to_string())
    return res


# --- run it -----------------------------------------------------------------
# ABLATION = rate_ablation(strokes, qc, mcfg, epochs=30)

---
## 8.6 Save, and score a new session

`score_session()` closes the loop: raw CSV → clean → segment → **pack raw
points** → scale → windows → embedding → similarity against every enrolled user.
Same code path as training, which is the point of saving `seq_scaler` and
`SeqConfig` alongside the weights: a channel order or a `max_len` that disagrees
with training silently produces a different feature space, and nothing will raise.

For an authentication decision rather than a ranking, use the threshold in
`res["test_session"]["thr"]` — the EER operating point. Move it up for fewer false
accepts, down for fewer false rejects; there is no symmetric choice that is
automatically right.

In [ ]:
def save_model(outdir, enc, bank, mcfg, scfg, metrics, hist, z_enroll, y_enroll, tag):
    out = Path(outdir) / tag
    out.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": enc.state_dict(), "kind": bank.kind, "n_ch": enc.n_ch,
                "model_config": asdict(mcfg), "seq_config": asdict(scfg)},
               out / "encoder.pt")
    T, us = templates(z_enroll, y_enroll)
    np.savez(out / "templates.npz", T=T, users=us)
    pd.DataFrame(hist).to_csv(out / "history.csv", index=False)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))
    # the seq scaler for the raw model, the tabular preprocessor for the baseline --
    # whichever one turns a session into what this encoder was trained on
    joblib.dump(bank.preproc, out / "preprocessor.joblib")
    print(f"[save] -> {out}  ({len(us)} enrolled users, {T.shape[1]}-d embeddings)")
    return out


def load_model(outdir, tag):
    out = Path(outdir) / tag
    ck = torch.load(out / "encoder.pt", map_location=DEVICE, weights_only=False)
    mcfg = ModelConfig(**ck["model_config"])
    scfg = SeqConfig(**ck["seq_config"])
    cls = {"raw": RawWindowEncoder, "features": FeatureWindowEncoder}[ck["kind"]]
    enc = cls(ck["n_ch"], mcfg).to(DEVICE)
    enc.load_state_dict(ck["state_dict"]); enc.eval()
    t = np.load(out / "templates.npz", allow_pickle=True)
    return enc, ck["kind"], mcfg, scfg, joblib.load(out / "preprocessor.joblib"), t["T"], t["users"]


def score_session(raw_csv_or_df, enc, kind, preproc, mcfg, scfg, T, users,
                  cfg: Config = Config(), pcfg: PrepConfig = None):
    """Raw session -> one embedding -> similarity to every enrolled user.

    Same chain as training, branching only where the two representations differ.
    Saving the preprocessor beside the weights is the point: a channel order, a
    max_len or a feature list that disagrees with training silently produces a
    different input space, and nothing raises.
    """
    pcfg = pcfg or PREP
    raw = pd.read_csv(raw_csv_or_df) if isinstance(raw_csv_or_df, (str, Path)) else raw_csv_or_df
    st, qc, seqs = process_frame(raw, "probe", "probe", replace(cfg, verbose=False), pcfg)
    if not len(st):
        raise RuntimeError(f"session rejected by QC: {qc.get('drop_reason')}")
    st = filter_strokes(st, pcfg)
    st["fold"] = "probe"

    if kind == "raw":
        Sp, lp, st = build_stroke_bank(st, seqs, scfg)
        Wp, yp, sp, fp, _ = build_window_index(st, mcfg)
        probe = WindowBank(Sp, lp, Wp, yp, sp, fp, scaler=preproc)
    else:
        st = st.reset_index(drop=True)
        Wp, yp, sp, fp, _ = build_window_index(st, mcfg)
        Vp = apply_preprocessor(st, preproc).to_numpy(dtype=np.float32)
        probe = FeatureBank(Vp, Wp, yp, sp, fp, preproc)

    z = embed(enc, probe, bs=mcfg.embed_bs)
    zs = z.mean(0); zs /= np.linalg.norm(zs)
    rank = (pd.DataFrame({"user": users, "similarity": T @ zs})
              .sort_values("similarity", ascending=False).reset_index(drop=True))
    return rank, zs, len(z)


MODEL_ROOT = OUT_ROOT / "models"        # ./data/models     |  Drive/trace-data/models

for name, bk in BANKS.items():
    z_enroll = embed(models[name], bk, idx["train"], mcfg.embed_bs)
    save_model(MODEL_ROOT, models[name], bk, mcfg, scfg, res[name], hists[name],
               z_enroll, y[idx["train"]], tag=f"siamese_{name}_v2")

# --- later / elsewhere -------------------------------------------------------
# enc2, kind, mcfg2, scfg2, pre2, T, users = load_model(MODEL_ROOT, "siamese_raw_v2")
# rank, z_sess, n_win = score_session(some_csv_path, enc2, kind, pre2, mcfg2, scfg2, T, users)
# display(rank.head())